# SQL 기초 — 통합 실습 노트북

설치가 필요한 것은 없습니다 — Python 내장 `sqlite3`만 씁니다.

## 교시 ↔ 섹션 대응표

| 교시 | 섹션 |
|---|---|
| 1 | DB API 손풀기 — cursor·execute·fetch |
| 2 | SELECT 기초와 WHERE |
| 3 | ORDER BY·LIMIT + INSERT·UPDATE·DELETE |
| 4 | 집계 함수와 GROUP BY·HAVING |
| 5 | JOIN 원리 — INNER JOIN |
| 6 | LEFT JOIN과 인덱스 |
| 7 | 종합 실습 — 쇼핑몰 DB(shop.db)를 처음부터 (CP1~CP5) |
| 8 | 개인 실습 — 한빛은행(hanbit_bank.db) |

> ⚠️ **위에서 아래로 순서대로 실행하세요.** 특히 7교시(CP1~CP5)는 **하나로 이어진 하나의 shop.db 세션**입니다 —
> 중간에 표를 다시 만들거나 커널을 재시작하면 안 됩니다. CP3에서 바꾼 텀블러 가격(25000원)이
> CP4의 금액 계산에 그대로 반영되는 것이 이 통합의 핵심 포인트입니다 (교안 7교시 "흔한 실수" 참고).
>
> 8교시(한빛은행)는 **직접 SQL을 작성하는 개인 실습**입니다 — 🧪 문제 → 💡 힌트 → (주석 처리된) 정답
> 순서로 구성되어 있으니, 힌트까지 보고 막히면 정답 셀의 주석을 풀어 확인하세요.


---
# 1교시 · DB API 손풀기 — cursor·execute·fetch

`SQL_기초.md` 1교시 = 아래 섹션 전체.

## 아주 작은 표로 먼저 감 잡기

처음부터 여러 표와 긴 SQL을 보지 않습니다. 여기서는 `snacks`라는 3행짜리 표 하나만 만들고, SQL이 어떤 식으로 데이터를 꺼내는지 확인합니다.

- **표(table)**: 엑셀 시트처럼 행과 열이 있는 데이터 묶음
- **행(row)**: 데이터 한 건
- **열(column)**: 데이터의 항목 이름
- **SELECT**: 표에서 데이터를 꺼내는 SQL 명령


In [55]:
# 연습용 작은 DB를 만듭니다. (여기 쓴 명령은 바로 아래 절에서 하나씩 배워요)
import sqlite3

practice_conn = sqlite3.connect("snack.db")
practice_cur = practice_conn.cursor()

# 표를 만들고 과자 3개를 넣습니다 — 전부 가장 기본 명령인 execute 하나로!
practice_cur.execute("DROP TABLE IF EXISTS snacks")
practice_cur.execute("""
CREATE TABLE snacks (
    id INTEGER PRIMARY KEY,
    name TEXT,
    price INTEGER
)
""")
practice_cur.execute("INSERT INTO snacks (id, name, price) VALUES (1, '새우깡', 1500)")
practice_cur.execute("INSERT INTO snacks (id, name, price) VALUES (2, '초코파이', 4200)")
practice_cur.execute("INSERT INTO snacks (id, name, price) VALUES (3, '감자칩', 1800)")
practice_conn.commit()

print("snacks 표 준비 완료: 과자 3개")

snacks 표 준비 완료: 과자 3개


### 전체 보기

`SELECT *`는 표의 **모든 열**을 보여 달라는 뜻입니다. `FROM snacks`는 `snacks` 표에서 가져오라는 뜻입니다.


In [56]:
rows = practice_cur.execute("SELECT * FROM snacks WHERE price > 2000").fetchall()

for row in rows:
    print(row)


(2, '초코파이', 4200)


### 필요한 열만 보기

`*` 대신 열 이름을 쓰면 필요한 열만 볼 수 있습니다. 아래 SQL은 과자 이름과 가격만 꺼냅니다.


In [57]:
rows = practice_cur.execute("SELECT name, price FROM snacks").fetchall()

for row in rows:
    print(row)


('새우깡', 1500)
('초코파이', 4200)
('감자칩', 1800)


### 조건으로 고르기

`WHERE`는 조건에 맞는 행만 고릅니다. 아래 SQL은 가격이 2,000원 이하인 과자만 보여 줍니다.


In [58]:
rows = practice_cur.execute("SELECT name, price FROM snacks WHERE price <= 2000").fetchall()

for row in rows:
    print(row)


('새우깡', 1500)
('감자칩', 1800)


### 여기까지의 핵심

SQL 조회는 우선 이 모양만 기억하면 됩니다.

```sql
SELECT 보고_싶은_열
FROM 표_이름
WHERE 조건;
```

이제 같은 원리를 조금 더 실제적인 도서관 DB에 적용합니다.


## Python에서 SQL 실행하기: cursor·execute·fetch

이번에는 SQL 자체보다 **Python이 SQL을 DB에 보내고 결과를 받는 방식**을 연습합니다.

- `cursor`: SQL을 실행하는 손잡이
- `execute`: SQL 한 문장을 실행
- `fetchall`: 결과를 모두 가져오기
- `fetchone`: 결과를 한 건만 가져오기
- `executemany`: 같은 SQL을 값 리스트로 반복 실행
- `executescript`: 여러 문장을 한 번에 실행
- placeholder: SQL 문자열에 값을 직접 붙이지 않고 안전하게 전달하기

Python 내장 `sqlite3`에서는 값이 들어갈 자리에 `?`를 쓰고, 실제 값은 `execute()`의 두 번째 인자로 전달합니다.

```python
cursor.execute("SELECT * FROM scores WHERE name=?", ("홍길동",))
```


### 1. cursor와 execute로 표 만들기

`CREATE TABLE`은 새 표를 만드는 SQL입니다. 여기서는 학생 점수를 담는 `scores` 표를 만듭니다.


In [59]:
# 앞에서 만든 practice DB 연결을 계속 사용합니다.
# 이 셀은 여러 번 실행해도 같은 결과가 나오도록 기존 scores 표를 지우고 다시 만듭니다.
practice_cur.execute("DROP TABLE IF EXISTS scores")

practice_cur.execute("""
CREATE TABLE scores (
    name TEXT PRIMARY KEY,
    math INTEGER,
    science INTEGER,
    english INTEGER
)
""")

practice_conn.commit()
print("scores 표 생성 완료")


scores 표 생성 완료


### 2. INSERT와 placeholder로 데이터 넣기

사용자가 입력한 값은 SQL 문장 안에 직접 끼워 넣지 않고 placeholder로 전달합니다. SQLite에서는 값이 들어갈 자리에 `?`를 씁니다.


In [60]:
# INSERT: scores 표에 한 학생의 점수를 넣습니다.
# ? 자리에 두 번째 인자로 넘긴 튜플 값들이 순서대로 들어갑니다.
practice_cur.execute(
    "INSERT INTO scores (name, math, science, english) VALUES (?, ?, ?, ?)",
    ("홍길동", 30, 20, 10),
)

practice_cur.execute(
    "INSERT INTO scores (name, math, science, english) VALUES (?, ?, ?, ?)",
    ("김영희", 80, 90, 85),
)

practice_conn.commit()
print("학생 점수 2건 입력 완료")


학생 점수 2건 입력 완료


### 3. fetchall로 전체 결과 가져오기

`fetchall()`은 조회 결과를 리스트처럼 한 번에 모두 가져옵니다.


In [61]:
practice_cur.execute("SELECT name, math, science, english FROM scores")
rows = practice_cur.fetchall()

for row in rows:
    print(row)


('홍길동', 30, 20, 10)
('김영희', 80, 90, 85)


### 4. fetchone으로 한 건만 가져오기

`fetchone()`은 조회 결과 중 첫 번째 한 건만 가져옵니다. 조건에 맞는 데이터가 없으면 `None`이 나옵니다.


In [62]:
practice_cur.execute(
    "SELECT name, math, science, english FROM scores WHERE name=?",
    ("홍길동",),
)
row = practice_cur.fetchone()

print(row)


('홍길동', 30, 20, 10)


### 5. DELETE와 placeholder로 데이터 삭제하기

`DELETE`도 조건값을 placeholder로 넘깁니다. 실수로 전체가 지워지지 않도록 보통 `WHERE`와 함께 씁니다.


In [63]:
practice_cur.execute(
    "DELETE FROM scores WHERE name=?",
    ("홍길동",),
)
practice_conn.commit()

practice_cur.execute("SELECT name, math, science, english FROM scores")
rows = practice_cur.fetchall()

print("삭제 후 남은 데이터")
for row in rows:
    print(row)


삭제 후 남은 데이터
('김영희', 80, 90, 85)


### 6. executemany로 여러 행을 한 번에 넣기

`execute`는 SQL **한 문장**을 실행합니다. 같은 INSERT를 행마다 반복해야 한다면, `executemany`가 **같은 SQL 하나 + 값 튜플 리스트**를 받아 알아서 반복해 줍니다.

In [64]:
# ### 2에서는 execute를 두 번 불러 두 명을 넣었죠 — executemany는 값 '리스트'로 한 번에!
practice_cur.executemany(
    "INSERT INTO scores (name, math, science, english) VALUES (?, ?, ?, ?)",
    [
        ("박철수", 70, 65, 90),
        ("이민지", 95, 88, 76),
    ],
)
practice_conn.commit()

practice_cur.execute("SELECT name, math, science, english FROM scores")
for row in practice_cur.fetchall():
    print(row)

('김영희', 80, 90, 85)
('박철수', 70, 65, 90)
('이민지', 95, 88, 76)


### 7. executescript로 여러 문장을 한 번에 실행하기

`executemany`가 '같은 문장 × 여러 값'이라면, `executescript`는 **서로 다른 문장 여러 개**를 세미콜론(`;`)으로 이어 한 번에 실행합니다. 표를 지우고 다시 만드는 초기화 스크립트에 딱이에요. (값을 바꿔 넣는 placeholder `?`는 못 씁니다 — 고정 스크립트 전용)

In [65]:
# 연습 표 scores를 스크립트 한 번으로 리셋합니다 — DROP + CREATE + INSERT 세 문장이 한 덩어리!
practice_cur.executescript("""
DROP TABLE IF EXISTS scores;

CREATE TABLE scores (
    name TEXT PRIMARY KEY,
    math INTEGER,
    science INTEGER,
    english INTEGER
);

INSERT INTO scores (name, math, science, english) VALUES ('홍길동', 30, 20, 10);
""")
practice_conn.commit()

practice_cur.execute("SELECT name, math, science, english FROM scores")
print(practice_cur.fetchall())

[('홍길동', 30, 20, 10)]


### DB API 손풀기 정리

- `conn = sqlite3.connect(...)`: DB 연결
- `cur = conn.cursor()`: SQL 실행 손잡이 만들기
- `cur.execute(sql, 값)`: SQL **한 문장** 실행
- `cur.executemany(sql, 값 리스트)`: 같은 SQL을 **여러 값으로 반복** 실행
- `cur.executescript("문장; 문장; ...")`: **여러 문장**을 한 번에 (초기화 스크립트)
- `fetchall()`: 결과 전체 가져오기
- `fetchone()`: 결과 한 건 가져오기
- SQLite placeholder는 `?`
- `INSERT`·`DELETE`처럼 데이터를 바꾸는 SQL 뒤에는 `commit()`으로 저장 확정

이제 같은 흐름을 조금 더 큰 도서관 DB에 적용합니다.


In [66]:
# ✅ 설치 없이 실행(sqlite3 내장) — pip install·API 키·외부 파일이 전혀 필요 없습니다.
# 💡 sqlite3는 Python '표준 라이브러리'라 import만 하면 바로 씁니다.
#    (SQLite = 파일 하나가 곧 DB인 초경량 RDBMS. Python에 내장되어 설치가 0입니다.)
import sqlite3

# 💡 sqlite3.sqlite_version은 실제로 SQL을 처리하는 SQLite 엔진의 버전입니다.
print("SQLite 엔진 버전  :", sqlite3.sqlite_version)


SQLite 엔진 버전  : 3.50.4


## 환경·DB 만들기 + SELECT 기초

- `import sqlite3` → `connect` → `cursor`로 DB에 연결합니다.
- 표 3개(`books`·`members`·`rentals`)를 **셀 안에서 인라인 생성**하고 시드 데이터를 넣습니다. (외부 `.db` 파일 없이 자기완결)
- `SELECT`로 표에서 데이터를 꺼냅니다 — 전체(`SELECT *`)와 특정 열(`SELECT title, author`).

In [67]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: connect → cursor → execute → fetch 가 sqlite3의 기본 4단계입니다.
# 💡 connect("library.db"): 'library.db'라는 파일 하나가 곧 데이터베이스입니다. (파일이 없으면 새로 만듭니다)
conn = sqlite3.connect("library.db")   # DB에 연결 (없으면 생성)
cur = conn.cursor()                    # 커서 = DB에 SQL을 실행하고 결과를 받아오는 '손잡이'
print("연결 완료 →", conn)

연결 완료 → <sqlite3.Connection object at 0x0000017E9EF66E30>


In [68]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: 'DROP TABLE IF EXISTS' 후 'CREATE TABLE' → 이 셀을 여러 번 실행해도 항상 같은 상태가 됩니다(재실행 안전).
# 💡 executescript: 여러 SQL 문장을 세미콜론(;)으로 이어 한 번에 실행합니다.
# 💡 id INTEGER PRIMARY KEY = 기본키(각 행을 유일하게 구분). FOREIGN KEY = 외래키(다른 표를 가리켜 연결).
#    (PostgreSQL 이식을 고려해 AUTOINCREMENT 없이 id를 직접 지정합니다.)
cur.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS members;
DROP TABLE IF EXISTS rentals;

CREATE TABLE books (
    id INTEGER PRIMARY KEY, title TEXT, author TEXT, year INTEGER, genre TEXT
);
CREATE TABLE members (
    id INTEGER PRIMARY KEY, name TEXT, email TEXT, join_date TEXT
);
CREATE TABLE rentals (
    id INTEGER PRIMARY KEY, book_id INTEGER, member_id INTEGER,
    rental_date TEXT, return_date TEXT,
    FOREIGN KEY(book_id) REFERENCES books(id),
    FOREIGN KEY(member_id) REFERENCES members(id)
);
""")
print("표 3개 생성 완료 → books, members, rentals")

표 3개 생성 완료 → books, members, rentals


**🗺️ 방금 만든 표 3개의 ERD** — `rentals`의 외래키 두 개가 `books`·`members`의 기본키를 가리킵니다.

![ERD](attachment:erd.png)

In [69]:
# 🧪 실습 — `members` 표를 만드세요 — id(정수·기본키) · name · email · join_date(모두 문자)
# 💡 힌트: CREATE TABLE 표이름 ( 열이름 자료형 [제약조건], ... )
# 👇 아래 빈 셀에 직접 작성해 보세요

In [70]:
cur.execute("CREATE TABLE members (id INTEGER PRIMARY KEY, name TEXT, email TEXT, join_date TEXT);")

OperationalError: table members already exists

In [ ]:
# 앞 셀에서 이미 만들었으니 DROP 후 다시 만듭니다 (재실행 안전)
cur.execute("DROP TABLE IF EXISTS members")
cur.execute("CREATE TABLE members (id INTEGER PRIMARY KEY, name TEXT, email TEXT, join_date TEXT);") #내가 작성한 코드
# cur.execute("""CREATE TABLE members (   #원래 적혀 있던 코드
#     id        INTEGER PRIMARY KEY,
#     name      TEXT,
#     email     TEXT,
#     join_date TEXT
# )""")
conn.commit()
# (run 헬퍼는 뒤에서 정의되므로 여기서는 print 로 확인합니다)
print(cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
# → [('books',), ('rentals',), ('members',)]
#   (members 를 다시 만들었으므로 목록 맨 뒤로 갑니다)

[('books',), ('rentals',), ('members',)]


In [ ]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: executemany로 여러 행을 한 번에 삽입하고, commit으로 '저장 확정'합니다.
# 💡 return_date=None(=SQL의 NULL)이면 '아직 반납 안 함 = 대여 중'.
books = [
    (1, "The Catcher in the Rye", "J.D. Salinger", 1951, "Fiction"),
    (2, "To Kill a Mockingbird", "Harper Lee", 1960, "Fiction"),
    (3, "1984", "George Orwell", 1949, "Dystopian"),
    (4, "Moby-Dick", "Herman Melville", 1851, "Adventure"),
    (5, "Animal Farm", "George Orwell", 1945, "Dystopian"),
    (6, "The Old Man and the Sea", "Ernest Hemingway", 1952, "Adventure"),
]
members = [
    (1, "Alice", "alice@example.com", "20240101"),
    (2, "Bob", "bob@example.com", "20240102"),
    (3, "Charlie", "charlie@example.com", "20240103"),
    (4, "David", "david@example.com", "20240104"),
    (5, "Emma", "emma@example.com", "20240215"),
]
rentals = [
    (1, 1, 1, "20240201", "20240215"), (2, 3, 1, "20240301", None),
    (3, 2, 2, "20240202", "20240216"), (4, 1, 3, "20240305", "20240320"),
    (5, 4, 4, "20240204", "20240218"), (6, 5, 2, "20240310", None),
    (7, 3, 4, "20240312", "20240401"), (8, 1, 2, "20240401", None),
    (9, 6, 1, "20240405", "20240420"), (10, 2, 3, "20240410", None),
]
cur.executemany("INSERT INTO books   VALUES (?, ?, ?, ?, ?)", books)
cur.executemany("INSERT INTO members VALUES (?, ?, ?, ?)", members)
cur.executemany("INSERT INTO rentals VALUES (?, ?, ?, ?, ?)", rentals)
conn.commit()   # ✅ 포인트: commit = 변경을 실제로 '저장 확정' (안 하면 저장되지 않습니다)
print(f"시드 삽입 완료 → books {len(books)} · members {len(members)} · rentals {len(rentals)}")

시드 삽입 완료 → books 6 · members 5 · rentals 10


In [ ]:
# 🧪 실습 — members에 회원 한 명을 넣고 확인하세요 — 6 · 'Frank' · 'frank@example.com' · '20240301'
# 💡 힌트: INSERT INTO 표 (열들) VALUES (값들) → commit() → SELECT 로 확인
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
cur.execute("""INSERT INTO members (id, name, email, join_date)
               VALUES (6, 'Frank', 'frank@example.com', '20240301')""")
conn.commit()
print(cur.execute("SELECT * FROM members WHERE id = 6").fetchall())
# → [(6, 'Frank', 'frank@example.com', '20240301')]

# 확인했으면 되돌립니다 — 뒤에 같은 회원을 다시 넣는 셀이 있어요
cur.execute("DELETE FROM members WHERE id = 6")
conn.commit()

[(6, 'Frank', 'frank@example.com', '20240301')]


In [ ]:
# ✅ 포인트: 조회 결과를 보기 좋게 출력하는 헬퍼 run()을 정의합니다. (pandas 없이 순수 파이썬)
# 💡 cur.execute(sql)로 SQL을 실행하고, fetchall()로 결과 행을 모두 가져옵니다.
def run(sql):
    """SQL을 실행하고 결과 행을 한 줄씩 출력한 뒤, 행 리스트를 돌려줍니다."""
    rows = cur.execute(sql).fetchall()   # 실행 후 모든 결과 행을 리스트로
    for r in rows:
        print(r)
    print(f"  → {len(rows)}행")
    return rows

# 💡 (선택) 표를 예쁘게 보고 싶다면 pandas를 설치해도 됩니다 — 설치가 필요한 선택 경로입니다:
#     %pip install -q pandas
#     import pandas as pd; pd.read_sql("SELECT * FROM books", conn)
print("헬퍼 run() 준비 완료")

헬퍼 run() 준비 완료


In [ ]:
# ✅ 포인트: SELECT는 '표에서 꺼내기'. SELECT * = 모든 열, FROM books = books 표에서.
# 전체 조회 — books 표의 모든 행·열을 꺼냅니다. (기대: 6행)
run("SELECT * FROM books")

(1, 'The Catcher in the Rye', 'J.D. Salinger', 1951, 'Fiction')
(2, 'To Kill a Mockingbird', 'Harper Lee', 1960, 'Fiction')
(3, '1984', 'George Orwell', 1949, 'Dystopian')
(4, 'Moby-Dick', 'Herman Melville', 1851, 'Adventure')
(5, 'Animal Farm', 'George Orwell', 1945, 'Dystopian')
(6, 'The Old Man and the Sea', 'Ernest Hemingway', 1952, 'Adventure')
  → 6행


[(1, 'The Catcher in the Rye', 'J.D. Salinger', 1951, 'Fiction'),
 (2, 'To Kill a Mockingbird', 'Harper Lee', 1960, 'Fiction'),
 (3, '1984', 'George Orwell', 1949, 'Dystopian'),
 (4, 'Moby-Dick', 'Herman Melville', 1851, 'Adventure'),
 (5, 'Animal Farm', 'George Orwell', 1945, 'Dystopian'),
 (6, 'The Old Man and the Sea', 'Ernest Hemingway', 1952, 'Adventure')]

In [ ]:
# ✅ 포인트: 필요한 열만 골라 꺼낼 수도 있습니다 (title, author 두 열만).
run("SELECT title, author FROM books")

('The Catcher in the Rye', 'J.D. Salinger')
('To Kill a Mockingbird', 'Harper Lee')
('1984', 'George Orwell')
('Moby-Dick', 'Herman Melville')
('Animal Farm', 'George Orwell')
('The Old Man and the Sea', 'Ernest Hemingway')
  → 6행


[('The Catcher in the Rye', 'J.D. Salinger'),
 ('To Kill a Mockingbird', 'Harper Lee'),
 ('1984', 'George Orwell'),
 ('Moby-Dick', 'Herman Melville'),
 ('Animal Farm', 'George Orwell'),
 ('The Old Man and the Sea', 'Ernest Hemingway')]

In [ ]:
# 🧪 실습 — books에서 제목(title)과 출간연도(year)만 조회하세요
# 💡 힌트: SELECT 열1, 열2 FROM 표이름
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("SELECT title, year FROM books")
# # → 6행 (열만 골랐을 뿐 행 수는 그대로)

In [ ]:
# 🧪 실습 — books의 저자(author)를 중복 없이 조회하세요
# 💡 힌트: SELECT DISTINCT 열이름 FROM 표이름
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("SELECT DISTINCT author FROM books")
# # → 5행 (책은 6권인데 저자는 5명 — George Orwell이 2권)

### SELECT 기초 정리
- **핵심**: `sqlite3.connect` → `cursor` → `execute` → `fetch`. `SELECT 열 FROM 표`로 데이터를 꺼냅니다.
- **흔한 실수**: 셋업 셀을 실행하지 않고 조회부터 하면 `no such table: books` 에러가 납니다 — 위에서부터 순서대로 실행하세요.

---
# 2교시 · SELECT 기초와 WHERE — 필요한 것만 고르기

## WHERE·ORDER BY·LIMIT — 고르고·정렬하고·자르기

- `WHERE`: 조건으로 **행을 고릅니다** (`=`·`>`·`<`·`>=`·`<=`·`!=`·`AND`·`OR`·`LIKE`).
- `ORDER BY`: 결과를 **정렬**합니다 (`ASC` 오름차순 / `DESC` 내림차순).
- `LIMIT`: 위에서 **N개만** 자릅니다.
- 흐름: **고르기(WHERE) → 줄세우기(ORDER BY) → 자르기(LIMIT)** 순으로 데이터를 좁혀 갑니다.

In [ ]:
# ✅ 포인트: WHERE=조건으로 고르기, ORDER BY=정렬(기본은 오름차순 ASC). 문자열 조건은 작은따옴표 ''.
# 장르가 Fiction인 책을 출간연도 순으로. (기대: Catcher 1951 → Mockingbird 1960)
run("SELECT title, year FROM books WHERE genre='Fiction' ORDER BY year")

('The Catcher in the Rye', 1951)
('To Kill a Mockingbird', 1960)
  → 2행


[('The Catcher in the Rye', 1951), ('To Kill a Mockingbird', 1960)]

In [ ]:
# ✅ 포인트: 조건은 =, >, <, >=, <=, !=, AND, OR, LIKE 로 다양하게 조합합니다.
print("[1950년 이후 출간]")
run("SELECT title, year FROM books WHERE year >= 1950")
print("\n[저자가 George Orwell]")
run("SELECT title FROM books WHERE author='George Orwell'")
print("\n[제목에 the 포함 — LIKE, %는 '아무 글자들'을 의미]")
run("SELECT title FROM books WHERE title LIKE '%the%'")

[1950년 이후 출간]
('The Catcher in the Rye', 1951)
('To Kill a Mockingbird', 1960)
('The Old Man and the Sea', 1952)
  → 3행

[저자가 George Orwell]
('1984',)
('Animal Farm',)
  → 2행

[제목에 the 포함 — LIKE, %는 '아무 글자들'을 의미]
('The Catcher in the Rye',)
('The Old Man and the Sea',)
  → 2행


[('The Catcher in the Rye',), ('The Old Man and the Sea',)]

In [ ]:
# ✅ 포인트: IN = 목록 중 하나(OR의 축약), IS NULL = '값이 없음' 찾기 (= NULL 은 동작하지 않습니다).
print("[장르가 Fiction 또는 Dystopian — IN은 목록 중 하나]")
run("SELECT title, genre FROM books WHERE genre IN ('Fiction', 'Dystopian')")

print("\n[아직 반납 안 한 대여 — IS NULL]")
run("SELECT id, book_id, member_id FROM rentals WHERE return_date IS NULL")


[장르가 Fiction 또는 Dystopian — IN은 목록 중 하나]
('The Catcher in the Rye', 'Fiction')
('To Kill a Mockingbird', 'Fiction')
('1984', 'Dystopian')
('Animal Farm', 'Dystopian')
  → 4행

[아직 반납 안 한 대여 — IS NULL]
(2, 3, 1)
(6, 5, 2)
(8, 1, 2)
(10, 2, 3)
  → 4행


[(2, 3, 1), (6, 5, 2), (8, 1, 2), (10, 2, 3)]

In [ ]:
# 🧪 실습 — 1950년 이후(1950년 포함) 출간된 책의 제목·연도를 조회하세요
# 💡 힌트: WHERE 열 >= 값   ('이후'는 그 해를 포함하므로 > 가 아니라 >=)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT title, year FROM books WHERE year >= 1950")
# # → Catcher(1951) · Mockingbird(1960) · Old Man(1952)  → 3행

('The Catcher in the Rye', 1951)
('To Kill a Mockingbird', 1960)
('The Old Man and the Sea', 1952)
  → 3행


[('The Catcher in the Rye', 1951),
 ('To Kill a Mockingbird', 1960),
 ('The Old Man and the Sea', 1952)]

In [ ]:
# 🧪 실습 — 장르가 Dystopian 이거나 Adventure인 책의 제목·장르를 조회하세요
# 💡 힌트: '이거나' = OR   (AND로 쓰면 0행 — 한 책의 장르는 하나뿐)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT title, genre FROM books
        WHERE genre = 'Dystopian' OR genre = 'Adventure'""")
# # → 4행

('1984', 'Dystopian')
('Moby-Dick', 'Adventure')
('Animal Farm', 'Dystopian')
('The Old Man and the Sea', 'Adventure')
  → 4행


[('1984', 'Dystopian'),
 ('Moby-Dick', 'Adventure'),
 ('Animal Farm', 'Dystopian'),
 ('The Old Man and the Sea', 'Adventure')]

In [ ]:
# 🧪 실습 — 저자 이름이 'George'로 시작하는 책의 제목·저자를 조회하세요
# 💡 힌트: LIKE 'George%'  —  % 를 뒤에만 붙이면 '시작', 양쪽에 붙이면 '포함'
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
run("SELECT title, author FROM books WHERE author LIKE 'George%'")

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT title, author FROM books WHERE author LIKE 'George%'")
# # → ('1984', 'George Orwell') · ('Animal Farm', 'George Orwell')  → 2행

('1984', 'George Orwell')
('Animal Farm', 'George Orwell')
  → 2행


[('1984', 'George Orwell'), ('Animal Farm', 'George Orwell')]

In [ ]:
# 🧪 실습 — 1949년부터 1952년 사이에 출간된 책의 제목·연도를 연도 오름차순으로 조회하세요
# 💡 힌트: BETWEEN 작은값 AND 큰값 (양끝 포함) + ORDER BY
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
run()

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT title, year FROM books
        WHERE year BETWEEN 1949 AND 1952
        ORDER BY year""")
# # → 1984(1949) · Catcher(1951) · Old Man(1952)  → 3행 (양끝 포함)

('1984', 1949)
('The Catcher in the Rye', 1951)
('The Old Man and the Sea', 1952)
  → 3행


[('1984', 1949),
 ('The Catcher in the Rye', 1951),
 ('The Old Man and the Sea', 1952)]

In [ ]:
# 🧪 실습 — 반납이 끝난(return_date가 있는) 대여 기록의 id와 반납일을 조회하세요
# 💡 힌트: IS NOT NULL  (= NULL / != NULL 은 동작하지 않습니다)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT id, return_date FROM rentals WHERE return_date IS NOT NULL")
# # → 6행 (반납 완료 6건 + 대여 중 4건 = 전체 10건)

(1, '20240215')
(3, '20240216')
(4, '20240320')
(5, '20240218')
(7, '20240401')
(9, '20240420')
  → 6행


[(1, '20240215'),
 (3, '20240216'),
 (4, '20240320'),
 (5, '20240218'),
 (7, '20240401'),
 (9, '20240420')]

---
# 3교시 · ORDER BY·LIMIT + INSERT·UPDATE·DELETE — 정렬·추출과 데이터 조작

In [ ]:
# ✅ 포인트: ORDER BY ... DESC = 내림차순, LIMIT N = 위에서 N개만.
# 가장 최근에 나온 책 3권. (기대: Mockingbird 1960 · Old Man 1952 · Catcher 1951)
run("SELECT title, year FROM books ORDER BY year DESC LIMIT 3")

('To Kill a Mockingbird', 1960)
('The Old Man and the Sea', 1952)
('The Catcher in the Rye', 1951)
  → 3행


[('To Kill a Mockingbird', 1960),
 ('The Old Man and the Sea', 1952),
 ('The Catcher in the Rye', 1951)]

In [ ]:
# 🧪 실습 — 최근에 나온 책부터 제목·연도를 조회하세요
# 💡 힌트: ORDER BY 열 DESC  (기본값은 ASC = 오름차순)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT title, year FROM books ORDER BY year DESC")
# # → Mockingbird(1960) 부터 Moby-Dick(1851) 까지  → 6행

('To Kill a Mockingbird', 1960)
('The Old Man and the Sea', 1952)
('The Catcher in the Rye', 1951)
('1984', 1949)
('Animal Farm', 1945)
('Moby-Dick', 1851)
  → 6행


[('To Kill a Mockingbird', 1960),
 ('The Old Man and the Sea', 1952),
 ('The Catcher in the Rye', 1951),
 ('1984', 1949),
 ('Animal Farm', 1945),
 ('Moby-Dick', 1851)]

In [ ]:
# 🧪 실습 — 가장 오래된 책 2권의 제목·연도를 조회하세요
# 💡 힌트: 정렬(ORDER BY)을 먼저 하고 LIMIT 2 로 자릅니다
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT title, year FROM books ORDER BY year LIMIT 2")
# # → Moby-Dick(1851) · Animal Farm(1945)  → 2행

('Moby-Dick', 1851)
('Animal Farm', 1945)
  → 2행


[('Moby-Dick', 1851), ('Animal Farm', 1945)]

### WHERE·ORDER BY·LIMIT 정리
- **핵심**: `WHERE`(고르기) → `ORDER BY`(줄세우기) → `LIMIT`(N개) 순서로 원하는 데이터에 좁혀 갑니다.
- **RDBMS 호환 팁**: 날짜는 `'YYYYMMDD'` **문자열**로 저장해 문자 비교(`WHERE join_date >= '20240103'`)합니다. SQLite 전용 날짜 함수를 쓰지 않아 다른 RDBMS로 옮겨도 이해하기 쉽습니다.
- **흔한 실수**: 문자열 값에 따옴표를 빼면(`genre=Fiction`) 에러가 납니다 — 문자열은 반드시 `'Fiction'`.

## INSERT·UPDATE·DELETE — 넣고·고치고·지우기

- `INSERT`: 새 행을 **추가**합니다.
- `UPDATE ... SET ... WHERE`: 기존 행을 **수정**합니다.
- `DELETE ... WHERE`: 행을 **삭제**합니다.
- `commit`: 변경을 **저장 확정**합니다. (트랜잭션은 여기서는 저장 확정 의미로만 다룹니다.)
- ⚠️ 이후 집계·JOIN 결과를 지키기 위해, 조작 데모는 **임시 회원 id=6(Frank)** 로만 하고 마지막에 원상 복구합니다.

In [ ]:
# ✅ 포인트: INSERT INTO 표(열...) VALUES (값...) = 새 행 추가. 변경 후에는 commit으로 저장 확정.
cur.execute("INSERT INTO members (id, name, email, join_date) VALUES (6, 'Frank', 'frank@example.com', '20240301')")
conn.commit()
print("[INSERT 후 — 새로 추가된 Frank]")
run("SELECT * FROM members WHERE id=6")

[INSERT 후 — 새로 추가된 Frank]
(6, 'Frank', 'frank@example.com', '20240301')
  → 1행


[(6, 'Frank', 'frank@example.com', '20240301')]

In [ ]:
# ✅ 포인트: UPDATE 표 SET 열=값 WHERE 조건 = 기존 행 수정. WHERE로 대상을 '반드시' 지정합니다.
cur.execute("UPDATE members SET email='frank2@example.com' WHERE id=6")
conn.commit()
print("[UPDATE 후 — Frank의 바뀐 이메일]")
run("SELECT name, email FROM members WHERE id=6")

[UPDATE 후 — Frank의 바뀐 이메일]
('Frank', 'frank2@example.com')
  → 1행


[('Frank', 'frank2@example.com')]

In [ ]:
# 🧪 실습 — id가 6인 회원의 이름을 'Frank Miller'로 바꾸고 확인하세요
# 💡 힌트: 확인(SELECT) → 수정(UPDATE ... WHERE) → commit → 재확인
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("SELECT id, name FROM members WHERE id = 6")      # ① 대상 확인
# cur.execute("UPDATE members SET name = 'Frank Miller' WHERE id = 6")
# conn.commit()                                        # ② 수정
# run("SELECT id, name FROM members WHERE id = 6")      # ③ 재확인
# # → (6, 'Frank Miller')

In [ ]:
# 🧪 실습 — id가 6인 회원을 지우고, 회원이 5명으로 돌아왔는지 확인하세요
# 💡 힌트: DELETE FROM 표 WHERE 조건 → commit → COUNT(*) 로 확인
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("SELECT id, name FROM members WHERE id = 6")   # ① 대상 확인
# cur.execute("DELETE FROM members WHERE id = 6")    # ② 삭제
# conn.commit()
# run("SELECT COUNT(*) FROM members")                # ③ → (5,)
#
# # ℹ️ 바로 다음 셀도 같은 일(시드 원복)을 합니다 —
# #    여기서 이미 지웠다면 다음 셀은 0행을 지우고 같은 (5,) 를 출력합니다.

In [ ]:
# ✅ 포인트: DELETE FROM 표 WHERE 조건 = 행 삭제. 임시 회원 Frank를 지워 시드를 원상 복구합니다.
# 💡 시드를 원래대로 되돌려야 이후 집계와 JOIN 결과가 확정값과 일치합니다.
cur.execute("DELETE FROM members WHERE id=6")
conn.commit()
print("[DELETE 후 — 회원 수는 다시 5명]")
run("SELECT COUNT(*) FROM members")

[DELETE 후 — 회원 수는 다시 5명]
(5,)
  → 1행


[(5,)]

### 넣고·고치고·지우기 정리
- **핵심**: `INSERT`(넣기)·`UPDATE`(고치기)·`DELETE`(지우기)는 반드시 `WHERE`로 대상을 지정하고 `commit`으로 저장 확정.
- ⚠️ **가장 위험한 실수 — `WHERE` 없는 UPDATE/DELETE**: `DELETE FROM members`(WHERE 없음)는 **모든 회원을 삭제**하고, `UPDATE members SET email='x'`는 **모든 회원의 이메일을 덮어씁니다**. 조작 문에는 `WHERE`가 있는지 항상 먼저 확인하세요. (이 노트북에서는 위험을 설명만 하고 실행하지 않습니다)
- **`commit`을 빠뜨리면** 변경이 저장되지 않습니다(조회는 commit 불필요, 변경에만 필요).

---
# 4교시 · 집계 함수와 GROUP BY·HAVING — 요약하기

## 집계 함수와 GROUP BY — 데이터 요약하기

- **집계 함수**(`COUNT`·`SUM`·`AVG`·`MIN`·`MAX`): 여러 행을 **하나의 값으로 요약**합니다.
- `GROUP BY`: **같은 값끼리 묶어** 그룹별로 집계합니다 (장르별 권수 등).
- `HAVING`: **집계한 뒤** 그룹에 조건을 겁니다 (`WHERE`는 집계 전, `HAVING`은 집계 후 — 맛보기).

In [ ]:
# ✅ 포인트: 집계 함수는 여러 행을 하나의 값으로 요약합니다 (COUNT·SUM·AVG·MIN·MAX).
# 전체 책 수 (기대: 6)
run("SELECT COUNT(*) FROM books")
print("\n[가장 오래된/최신/평균 출간연도]")
run("SELECT MIN(year), MAX(year), AVG(year) FROM books")

(6,)
  → 1행

[가장 오래된/최신/평균 출간연도]
(1851, 1960, 1934.6666666666667)
  → 1행


[(1851, 1960, 1934.6666666666667)]

In [ ]:
# 🧪 실습 — books의 권수와 평균 출간연도를 한 번에 조회하세요
# 💡 힌트: 집계 함수는 쉼표로 나란히 쓸 수 있습니다. ROUND(값, 자릿수) 로 소수점 정리
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT COUNT(*), AVG(year) FROM books")
# # → (6, 1934.6666666666667)
# run("SELECT COUNT(*), ROUND(AVG(year), 1) FROM books")
# # → (6, 1934.7)

(6, 1934.6666666666667)
  → 1행


[(6, 1934.6666666666667)]

In [ ]:
# 🧪 실습 — rentals에서 가장 이른 대여일과 가장 늦은 대여일을 조회하세요
# 💡 힌트: MIN·MAX 는 'YYYYMMDD' 문자열에도 통합니다 (자릿수를 맞춰 담았으므로)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("SELECT MIN(rental_date), MAX(rental_date) FROM rentals")
# # → ('20240201', '20240410')

('20240201', '20240410')
  → 1행


[('20240201', '20240410')]

In [ ]:
# ✅ 포인트: GROUP BY = 같은 값끼리 묶어 그룹별로 집계 (장르라는 '바구니'에 나눠 담고 개수 세기).
# 💡 SELECT의 열은 '집계 함수'이거나 'GROUP BY에 있는 열'만 두는 것이 RDBMS 전반에서 통하는 습관입니다.
# 장르별 책 권수 (기대: Fiction 2 · Dystopian 2 · Adventure 2)
run("SELECT genre, COUNT(*) AS cnt FROM books GROUP BY genre ORDER BY cnt DESC")
print("\n[저자별 권수 — George Orwell만 2권]")
run("SELECT author, COUNT(*) AS cnt FROM books GROUP BY author ORDER BY cnt DESC, author")

('Fiction', 2)
('Dystopian', 2)
('Adventure', 2)
  → 3행

[저자별 권수 — George Orwell만 2권]
('George Orwell', 2)
('Ernest Hemingway', 1)
('Harper Lee', 1)
('Herman Melville', 1)
('J.D. Salinger', 1)
  → 5행


[('George Orwell', 2),
 ('Ernest Hemingway', 1),
 ('Harper Lee', 1),
 ('Herman Melville', 1),
 ('J.D. Salinger', 1)]

In [ ]:
# 🧪 실습 — 저자별 권수를 '저자'·'권수'라는 이름으로, 권수가 많은 순으로 조회하세요
# 💡 힌트: AS 로 붙인 별칭은 ORDER BY 에서 그대로 쓸 수 있습니다
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
run("SELECT author AS 저자, COUNT(*) AS 권수 FROM books GROUP BY author ORDER BY 권수 desc")

('George Orwell', 2)
('J.D. Salinger', 1)
('Herman Melville', 1)
('Harper Lee', 1)
('Ernest Hemingway', 1)
  → 5행


[('George Orwell', 2),
 ('J.D. Salinger', 1),
 ('Herman Melville', 1),
 ('Harper Lee', 1),
 ('Ernest Hemingway', 1)]

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
cur.execute("""SELECT author AS 저자, COUNT(*) AS 권수
               FROM books GROUP BY author
               ORDER BY 권수 DESC, 저자""")
print([d[0] for d in cur.description])   # → ['저자', '권수']
for r in cur.fetchall():
    print(r)

['저자', '권수']
('George Orwell', 2)
('Ernest Hemingway', 1)
('Harper Lee', 1)
('Herman Melville', 1)
('J.D. Salinger', 1)


In [ ]:
# 🧪 실습 — rentals에서 회원별(member_id) 대여 건수를 많은 순으로 조회하세요
# 💡 힌트: GROUP BY 로 묶고 COUNT 로 셉니다. 동점이면 ORDER BY 에 열을 하나 더
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT member_id, COUNT(*) AS cnt FROM rentals
        GROUP BY member_id
        ORDER BY cnt DESC, member_id""")
# # → (1,3) (2,3) (3,2) (4,2)  → 4행
# # ⚠️ 5번 회원(Emma)은 대여가 없어 아예 안 나옵니다 → LEFT JOIN 편에서 해결

(1, 3)
(2, 3)
(3, 2)
(4, 2)
  → 4행


[(1, 3), (2, 3), (3, 2), (4, 2)]

In [ ]:
# ✅ 포인트: WHERE는 '집계 전' 행을 고르고, HAVING은 '집계 후' 그룹을 고릅니다.
# [HAVING 맛보기] 2권 이상인 장르만. (모든 장르가 2권이라 셋 다 통과)
run("SELECT genre, COUNT(*) AS cnt FROM books GROUP BY genre HAVING COUNT(*) >= 2 ORDER BY genre")

('Adventure', 2)
('Dystopian', 2)
('Fiction', 2)
  → 3행


[('Adventure', 2), ('Dystopian', 2), ('Fiction', 2)]

In [54]:
run("SELECT * FROM rentals")

(1, 1, 1, '20240201', '20240215')
(2, 3, 1, '20240301', None)
(3, 2, 2, '20240202', '20240216')
(4, 1, 3, '20240305', '20240320')
(5, 4, 4, '20240204', '20240218')
(6, 5, 2, '20240310', None)
(7, 3, 4, '20240312', '20240401')
(8, 1, 2, '20240401', None)
(9, 6, 1, '20240405', '20240420')
(10, 2, 3, '20240410', None)
  → 10행


[(1, 1, 1, '20240201', '20240215'),
 (2, 3, 1, '20240301', None),
 (3, 2, 2, '20240202', '20240216'),
 (4, 1, 3, '20240305', '20240320'),
 (5, 4, 4, '20240204', '20240218'),
 (6, 5, 2, '20240310', None),
 (7, 3, 4, '20240312', '20240401'),
 (8, 1, 2, '20240401', None),
 (9, 6, 1, '20240405', '20240420'),
 (10, 2, 3, '20240410', None)]

In [ ]:
# 🧪 실습 — rentals에서 3건 이상 빌린 회원(member_id)만 조회하세요
# 💡 힌트: 집계 결과에 거는 조건은 WHERE 가 아니라 HAVING
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
run("SELECT member_id FROM rentals GROUP BY member_id ")

In [52]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT member_id, COUNT(*) AS cnt FROM rentals
        GROUP BY member_id
        HAVING COUNT(*) >= 3""")
# # → (1,3) (2,3)  → 2행  (HAVING 없으면 4행)
#
# # ⚠️ 같은 조건을 WHERE 에 쓰면 오류 — 한 번 직접 실행해 보세요
# #    run("SELECT member_id, COUNT(*) FROM rentals WHERE COUNT(*) >= 3 GROUP BY member_id")
# #    OperationalError: misuse of aggregate: COUNT()

(1, 3)
(2, 3)
  → 2행


[(1, 3), (2, 3)]

In [ ]:
# 🧪 실습 — 종합 실습 — 배운 명령어를 조합해 보세요
# 💡 힌트: Q1 조건+정렬 / Q2 묶기+집계 후 조건 / Q3 NULL / Q4 묶고·세고·정렬하고·자르기
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# # Q1. 1950년 이후 출간된 책을 최신순으로 제목·연도
# run("SELECT title, year FROM books WHERE year >= 1950 ORDER BY year DESC")
#
# # Q2. 2권 이상 쓴 저자와 권수
# run("""SELECT author, COUNT(*) AS cnt FROM books
#         GROUP BY author HAVING COUNT(*) >= 2""")
#
# # Q3. 아직 반납 안 한 대여의 id·book_id·member_id
# run("SELECT id, book_id, member_id FROM rentals WHERE return_date IS NULL")
#
# # Q4. 대여가 가장 많은 회원 Top 2
# run("""SELECT member_id, COUNT(*) AS cnt FROM rentals
#         GROUP BY member_id
#         ORDER BY cnt DESC, member_id LIMIT 2""")

### 집계 함수와 GROUP BY 정리
- **핵심**: `WHERE`(행 고르기) → `GROUP BY`(묶기) → 집계 함수(요약) → `HAVING`(묶은 뒤 고르기).
- **흔한 실수 — `GROUP BY`에 없는 열을 `SELECT`**: 그룹으로 묶었다면 `SELECT`에는 **집계 함수**이거나 **`GROUP BY`에 넣은 열**만 와야 합니다. 묶이지 않은 일반 열(예: `title`)을 그대로 뽑으면 그룹당 어느 행의 값을 보여줄지 정해지지 않아, PostgreSQL에서는 에러가 납니다 (위 GROUP BY 셀의 호환성 주석과 같은 규칙).

In [71]:
# 실습을 마치며 연결을 닫습니다. (변경은 이미 commit으로 저장 확정되었습니다)
conn.close()
print("연결을 닫았습니다.")

연결을 닫았습니다.


## 마무리

도서관 `library` DB로 SQL의 **조회·조작·집계**를 손에 익혔습니다.
- **꺼내기**: `SELECT` / `WHERE`(고르기) / `ORDER BY`(정렬) / `LIMIT`(N개)
- **바꾸기**: `INSERT` / `UPDATE` / `DELETE` + `commit`(저장 확정) — 항상 `WHERE`로 대상 지정
- **요약하기**: `COUNT·SUM·AVG·MIN·MAX` / `GROUP BY`(묶기) / `HAVING`(묶은 뒤 조건)

여기서 사용한 SQL 패턴은 SQLite뿐 아니라 PostgreSQL 같은 RDBMS에서도 기본적으로 이어지는 문법입니다.


---
# 5교시 · JOIN 원리 — INNER JOIN

## 미니 실습 — 번호를 이름으로 바꿔 붙이기

JOIN을 바로 길게 쓰기 전에, 아주 작은 표 2개로 감을 잡습니다.

상황은 단순합니다.

- `mini_members`: 회원번호와 이름
- `mini_rentals`: 대여기록. 그런데 이름은 없고 `member_id` 번호만 있음

목표는 `member_id` 번호를 회원 이름으로 바꿔 붙여서 보는 것입니다.


In [72]:
# 아주 작은 연습용 DB입니다. 기존 library DB와 섞이지 않도록 메모리에 따로 만듭니다.
import sqlite3

mini_conn = sqlite3.connect(":memory:")
mini_cur = mini_conn.cursor()

mini_cur.executescript("""
CREATE TABLE mini_members (
    id INTEGER PRIMARY KEY,
    name TEXT
);

CREATE TABLE mini_rentals (
    id INTEGER PRIMARY KEY,
    member_id INTEGER,
    item TEXT
);
""")

mini_cur.executemany(
    "INSERT INTO mini_members (id, name) VALUES (?, ?)",
    [
        (1, "Alice"),
        (2, "Bob"),
    ],
)

mini_cur.executemany(
    "INSERT INTO mini_rentals (id, member_id, item) VALUES (?, ?, ?)",
    [
        (1, 1, "1984"),
        (2, 2, "Moby-Dick"),
    ],
)

mini_conn.commit()

print("[대여 기록 원본 — 아직 이름이 없고 member_id 번호만 보입니다]")
for row in mini_cur.execute("SELECT id, member_id, item FROM mini_rentals"):
    print(row)


[대여 기록 원본 — 아직 이름이 없고 member_id 번호만 보입니다]
(1, 1, '1984')
(2, 2, 'Moby-Dick')


In [73]:
# member_id 번호를 mini_members.id와 맞춰서 이름을 붙입니다.
# ON r.member_id = m.id 가 "대여 기록의 회원번호 = 회원 표의 id"라는 연결 조건입니다.
rows = mini_cur.execute("""
SELECT m.name, r.item
FROM mini_rentals r
JOIN mini_members m ON r.member_id = m.id
ORDER BY r.id
""").fetchall()

print("[JOIN 후 — 번호 대신 이름이 붙었습니다]")
for row in rows:
    print(row)


[JOIN 후 — 번호 대신 이름이 붙었습니다]
('Alice', '1984')
('Bob', 'Moby-Dick')


### 미니 실습 정리

`JOIN`은 어려운 새 마법이 아니라, **번호를 기준으로 다른 표의 정보를 찾아 붙이는 것**입니다.

```sql
FROM mini_rentals r
JOIN mini_members m ON r.member_id = m.id
```

이제 같은 원리를 도서관 DB의 세 표(`rentals`·`members`·`books`)에 적용합니다.


## 준비 — library DB 만들기

JOIN 예제를 위해 표 3개와 시드를 **다시 인라인 생성**합니다. 노트북을 위에서 아래로 실행하면 필요한 데이터가 먼저 준비됩니다.


In [74]:
# ✅ 설치 없이 실행(sqlite3 내장) — library DB를 재생성합니다(재실행 안전).
import sqlite3

conn = sqlite3.connect("library.db")   # 파일 하나가 곧 DB (없으면 생성)
cur = conn.cursor()
cur.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS members;
DROP TABLE IF EXISTS rentals;
CREATE TABLE books (
    id INTEGER PRIMARY KEY, title TEXT, author TEXT, year INTEGER, genre TEXT
);
CREATE TABLE members (
    id INTEGER PRIMARY KEY, name TEXT, email TEXT, join_date TEXT
);
CREATE TABLE rentals (
    id INTEGER PRIMARY KEY, book_id INTEGER, member_id INTEGER,
    rental_date TEXT, return_date TEXT,
    FOREIGN KEY(book_id) REFERENCES books(id),
    FOREIGN KEY(member_id) REFERENCES members(id)
);
""")
print("표 3개 생성 완료 → books, members, rentals")

표 3개 생성 완료 → books, members, rentals


**🗺️ 오늘 이어 붙일 표 3개의 ERD** — JOIN은 이 화살표(외래키 → 기본키)를 따라 붙입니다.

![ERD](attachment:erd.png)

In [75]:
# ✅ 설치 없이 실행 — 시드 삽입 후 commit, 그리고 조회 헬퍼 run()을 준비합니다.
# 💡 return_date=None(NULL)이면 '대여 중'. Emma(id=5)는 대여 기록이 0건입니다(LEFT JOIN 주인공).
books = [
    (1, "The Catcher in the Rye", "J.D. Salinger", 1951, "Fiction"),
    (2, "To Kill a Mockingbird", "Harper Lee", 1960, "Fiction"),
    (3, "1984", "George Orwell", 1949, "Dystopian"),
    (4, "Moby-Dick", "Herman Melville", 1851, "Adventure"),
    (5, "Animal Farm", "George Orwell", 1945, "Dystopian"),
    (6, "The Old Man and the Sea", "Ernest Hemingway", 1952, "Adventure"),
]
members = [
    (1, "Alice", "alice@example.com", "20240101"), (2, "Bob", "bob@example.com", "20240102"),
    (3, "Charlie", "charlie@example.com", "20240103"), (4, "David", "david@example.com", "20240104"),
    (5, "Emma", "emma@example.com", "20240215"),
]
rentals = [
    (1, 1, 1, "20240201", "20240215"), (2, 3, 1, "20240301", None),
    (3, 2, 2, "20240202", "20240216"), (4, 1, 3, "20240305", "20240320"),
    (5, 4, 4, "20240204", "20240218"), (6, 5, 2, "20240310", None),
    (7, 3, 4, "20240312", "20240401"), (8, 1, 2, "20240401", None),
    (9, 6, 1, "20240405", "20240420"), (10, 2, 3, "20240410", None),
]
cur.executemany("INSERT INTO books   VALUES (?, ?, ?, ?, ?)", books)
cur.executemany("INSERT INTO members VALUES (?, ?, ?, ?)", members)
cur.executemany("INSERT INTO rentals VALUES (?, ?, ?, ?, ?)", rentals)
conn.commit()

def run(sql):
    """SQL을 실행하고 결과 행을 한 줄씩 출력한 뒤, 행 리스트를 돌려줍니다."""
    rows = cur.execute(sql).fetchall()
    for r in rows:
        print(r)
    print(f"  → {len(rows)}행")
    return rows

print(f"시드 삽입 완료 → books {len(books)} · members {len(members)} · rentals {len(rentals)}")

시드 삽입 완료 → books 6 · members 5 · rentals 10


## JOIN 개념 — 왜 나눴다가 다시 잇나

- 우리는 중복을 줄이려 데이터를 **여러 표로 나눠** 담았습니다(정규화). 대여 장부(`rentals`)에는 **번호(book_id·member_id)만** 있고, 이름·제목은 `members`·`books`에 있습니다.
- "누가 무슨 책을 빌렸나"에 답하려면 **세 표를 다시 합쳐야** 합니다 — 그게 `JOIN`입니다.
- **INNER JOIN** = 두 표에서 **매칭되는 행만**(교집합). **`ON`** = 어떤 열로 잇나(외래키 = 기본키). **별칭(alias)** `r`·`m`·`b`로 표 이름을 줄입니다.
- 비유: 대여 장부엔 회원번호만 → **회원 명부·도서 목록에서 이름·제목을 '찾아 붙이는'** 것이 JOIN.

In [76]:
# ✅ 포인트: rentals 표에는 '번호'만 있습니다 — 누가·무슨 책인지 '이름'이 없습니다.
# 💡 이름은 members에, 제목은 books에 있죠. 번호로 '찾아 붙이는' 것이 바로 JOIN입니다.
print("[rentals 원본 — book_id·member_id 번호만 보임]")
run("SELECT id, book_id, member_id, return_date FROM rentals LIMIT 5")

[rentals 원본 — book_id·member_id 번호만 보임]
(1, 1, 1, '20240215')
(2, 3, 1, None)
(3, 2, 2, '20240216')
(4, 1, 3, '20240320')
(5, 4, 4, '20240218')
  → 5행


[(1, 1, 1, '20240215'),
 (2, 3, 1, None),
 (3, 2, 2, '20240216'),
 (4, 1, 3, '20240320'),
 (5, 4, 4, '20240218')]

## INNER JOIN — 먼저 한 번만 연결하기

세 표를 한 번에 잇기 전에, `rentals`와 `members`만 연결합니다.

- `rentals`: 대여 기록. `member_id` 번호만 있음
- `members`: 회원 번호와 이름이 있음
- 목표: `member_id` 번호를 회원 이름으로 바꿔 붙이기

이 단계에서는 책 제목까지 붙이지 않습니다. `book_id`는 일부러 번호 그대로 남겨 둡니다.


In [77]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: JOIN을 먼저 한 번만 써서 member_id 번호에 회원 이름을 붙입니다.
# 💡 아직 books는 JOIN하지 않았기 때문에 book_id는 번호 그대로 남아 있습니다.
run("""SELECT r.id, m.name, r.book_id
       FROM rentals r
       JOIN members m ON r.member_id = m.id
       ORDER BY r.id
       LIMIT 5""")


(1, 'Alice', 1)
(2, 'Alice', 3)
(3, 'Bob', 2)
(4, 'Charlie', 1)
(5, 'David', 4)
  → 5행


[(1, 'Alice', 1),
 (2, 'Alice', 3),
 (3, 'Bob', 2),
 (4, 'Charlie', 1),
 (5, 'David', 4)]

In [ ]:
# 🧪 실습 — rentals와 books를 이어 대여 id와 책 제목을 앞 5건 조회하세요
# 💡 힌트: JOIN 표B 별칭 ON 표A.외래키 = 표B.기본키  (여기서는 r.book_id = b.id)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [80]:
run("SELECT r.id, b.title FROM rentals r JOIN books b ON r.book_id  = b.id ORDER BY r.id LIMIT 5")

(1, 'The Catcher in the Rye')
(2, '1984')
(3, 'To Kill a Mockingbird')
(4, 'The Catcher in the Rye')
(5, 'Moby-Dick')
  → 5행


[(1, 'The Catcher in the Rye'),
 (2, '1984'),
 (3, 'To Kill a Mockingbird'),
 (4, 'The Catcher in the Rye'),
 (5, 'Moby-Dick')]

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("""SELECT r.id, b.title
#         FROM rentals r
#         JOIN books b ON r.book_id = b.id
#         ORDER BY r.id LIMIT 5""")
# # → (1,'The Catcher in the Rye') (2,'1984') (3,'To Kill a Mockingbird')
# #   (4,'The Catcher in the Rye') (5,'Moby-Dick')  → 5행

## INNER JOIN — 한 번 더 연결해 세 표로 확장하기

- 방금은 `rentals`와 `members`만 연결해서 회원 이름만 붙였습니다.
- 이제 `books`까지 한 번 더 연결해서 책 제목도 붙입니다.
- `INNER JOIN`은 매칭되는 행만 보여 줍니다. 대여 기록이 없는 **Emma**는 여기서는 나오지 않습니다.
- 이후 `LEFT JOIN`에서 Emma까지 포함하는 흐름을 비교합니다.


In [81]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: 방금 한 JOIN에 books를 한 번 더 붙여 책 제목까지 가져옵니다.
# 💡 r·m·b는 rentals·members·books의 별칭(alias) — 긴 표 이름을 짧게 씁니다.
# "누가 무슨 책을 빌렸나" — 회원 이름과 책 제목이 모두 붙습니다. (기대: 10행, Emma 없음)
run("""SELECT m.name, b.title
       FROM rentals r
       JOIN members m ON r.member_id = m.id
       JOIN books   b ON r.book_id   = b.id
       ORDER BY m.name, b.title""")


('Alice', '1984')
('Alice', 'The Catcher in the Rye')
('Alice', 'The Old Man and the Sea')
('Bob', 'Animal Farm')
('Bob', 'The Catcher in the Rye')
('Bob', 'To Kill a Mockingbird')
('Charlie', 'The Catcher in the Rye')
('Charlie', 'To Kill a Mockingbird')
('David', '1984')
('David', 'Moby-Dick')
  → 10행


[('Alice', '1984'),
 ('Alice', 'The Catcher in the Rye'),
 ('Alice', 'The Old Man and the Sea'),
 ('Bob', 'Animal Farm'),
 ('Bob', 'The Catcher in the Rye'),
 ('Bob', 'To Kill a Mockingbird'),
 ('Charlie', 'The Catcher in the Rye'),
 ('Charlie', 'To Kill a Mockingbird'),
 ('David', '1984'),
 ('David', 'Moby-Dick')]

In [ ]:
# 🧪 실습 — 회원 이름·책 제목·대여일을 대여일 순으로 앞 5건 조회하세요
# 💡 힌트: JOIN 을 두 번 붙입니다. rentals 가 members 와 books 를 잇는 다리
# 👇 아래 빈 셀에 직접 작성해 보세요

In [82]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT m.name, b.title, r.rental_date
        FROM rentals r
        JOIN members m ON r.member_id = m.id
        JOIN books   b ON r.book_id   = b.id
        ORDER BY r.rental_date LIMIT 5""")
# # → ('Alice','The Catcher in the Rye','20240201') ...  → 5행

('Alice', 'The Catcher in the Rye', '20240201')
('Bob', 'To Kill a Mockingbird', '20240202')
('David', 'Moby-Dick', '20240204')
('Alice', '1984', '20240301')
('Charlie', 'The Catcher in the Rye', '20240305')
  → 5행


[('Alice', 'The Catcher in the Rye', '20240201'),
 ('Bob', 'To Kill a Mockingbird', '20240202'),
 ('David', 'Moby-Dick', '20240204'),
 ('Alice', '1984', '20240301'),
 ('Charlie', 'The Catcher in the Rye', '20240305')]

In [83]:
# ✅ 포인트: JOIN한 결과를 GROUP BY로 묶어 '회원별 대여수'로 요약합니다.
# 회원별 대여 수 (INNER JOIN — 대여 기록이 있는 회원만). (기대: Alice 3·Bob 3·Charlie 2·David 2)
run("""SELECT m.name, COUNT(*) AS cnt
       FROM rentals r
       JOIN members m ON r.member_id = m.id
       GROUP BY m.name
       ORDER BY cnt DESC, m.name""")

('Alice', 3)
('Bob', 3)
('Charlie', 2)
('David', 2)
  → 4행


[('Alice', 3), ('Bob', 3), ('Charlie', 2), ('David', 2)]

In [84]:
# ✅ 포인트: 같은 방식으로 '인기 도서'(도서별 대여 수)도 뽑습니다.
# 인기 도서 Top. (기대: Catcher 3 · 1984 2 · Mockingbird 2 · 나머지 각 1)
run("""SELECT b.title, COUNT(*) AS cnt
       FROM rentals r
       JOIN books b ON r.book_id = b.id
       GROUP BY b.title
       ORDER BY cnt DESC, b.title""")

('The Catcher in the Rye', 3)
('1984', 2)
('To Kill a Mockingbird', 2)
('Animal Farm', 1)
('Moby-Dick', 1)
('The Old Man and the Sea', 1)
  → 6행


[('The Catcher in the Rye', 3),
 ('1984', 2),
 ('To Kill a Mockingbird', 2),
 ('Animal Farm', 1),
 ('Moby-Dick', 1),
 ('The Old Man and the Sea', 1)]

In [ ]:
# 🧪 실습 — 책별 대여 횟수를 제목과 함께 많은 순으로 조회하세요
# 💡 힌트: 먼저 JOIN 으로 제목을 붙이고, GROUP BY 로 묶습니다 (제목이 아니라 b.id 로 묶기)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [85]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
run("""SELECT b.title, COUNT(*) AS cnt
        FROM rentals r
        JOIN books b ON r.book_id = b.id
        GROUP BY b.id
        ORDER BY cnt DESC, b.title""")
# # → Catcher 3 · 1984 2 · Mockingbird 2 · 나머지 1씩  → 6행
# # 검산: 3+2+2+1+1+1 = 10 = 전체 대여 건수

('The Catcher in the Rye', 3)
('1984', 2)
('To Kill a Mockingbird', 2)
('Animal Farm', 1)
('Moby-Dick', 1)
('The Old Man and the Sea', 1)
  → 6행


[('The Catcher in the Rye', 3),
 ('1984', 2),
 ('To Kill a Mockingbird', 2),
 ('Animal Farm', 1),
 ('Moby-Dick', 1),
 ('The Old Man and the Sea', 1)]

---
# 6교시 · LEFT JOIN과 인덱스

### INNER JOIN vs LEFT JOIN

`INNER JOIN`은 **대여 기록이 있는 회원만** 나옵니다 — 대여가 0인 **Emma는 빠집니다.**
"대여를 한 번도 안 한 회원까지 포함해 세고 싶다"면 `LEFT JOIN`을 씁니다. 아래에서 **나란히** 비교합니다.

In [86]:
# ✅ 포인트: LEFT JOIN = 왼쪽 표(members)는 전부 + 매칭되는 오른쪽(rentals), 없으면 NULL.
# 💡 COUNT(r.id)는 NULL을 세지 않으므로, 대여가 0인 Emma는 0으로 나옵니다. (COUNT(*)는 모든 행을, COUNT(열)은 그 열이 NULL이 아닌 행만 셉니다.)
print("[INNER JOIN — 대여한 회원만 → Emma 없음]")
run("""SELECT m.name, COUNT(*) AS cnt
       FROM rentals r JOIN members m ON r.member_id = m.id
       GROUP BY m.name ORDER BY cnt DESC, m.name""")

print("\n[LEFT JOIN — 대여 0인 Emma까지 포함 ← 이 한 줄 차이가 LEFT JOIN의 존재 이유]")
run("""SELECT m.name, COUNT(r.id) AS cnt
       FROM members m LEFT JOIN rentals r ON m.id = r.member_id
       GROUP BY m.name ORDER BY cnt DESC, m.name""")

[INNER JOIN — 대여한 회원만 → Emma 없음]
('Alice', 3)
('Bob', 3)
('Charlie', 2)
('David', 2)
  → 4행

[LEFT JOIN — 대여 0인 Emma까지 포함 ← 이 한 줄 차이가 LEFT JOIN의 존재 이유]
('Alice', 3)
('Bob', 3)
('Charlie', 2)
('David', 2)
('Emma', 0)
  → 5행


[('Alice', 3), ('Bob', 3), ('Charlie', 2), ('David', 2), ('Emma', 0)]

In [ ]:
# 🧪 실습 — 모든 회원과 각자의 대여 건수를 많은 순으로 조회하세요 (대여가 없는 회원은 0)
# 💡 힌트: 함정 둘 — JOIN 이 아니라 LEFT JOIN, 그리고 COUNT(*) 가 아니라 COUNT(r.id)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# run("""SELECT m.name, COUNT(r.id) AS cnt
#         FROM members m
#         LEFT JOIN rentals r ON m.id = r.member_id
#         GROUP BY m.id
#         ORDER BY cnt DESC, m.name""")
# # → Alice 3 · Bob 3 · Charlie 2 · David 2 · Emma 0  → 5행
#
# # ⚠️ COUNT(*) 로 바꿔 실행해 보세요 — Emma 가 0 이 아니라 1 로 나옵니다
# #    (Emma 행은 존재하고 r 쪽 열만 NULL 이라, 행 수를 세면 1)

In [87]:
# ✅ 포인트: JOIN+GROUP BY 결과에도 HAVING으로 '집계 후' 조건을 겁니다.
# 3권 이상 빌린 회원만. (기대: Alice 3 · Bob 3)
run("""SELECT m.name, COUNT(*) AS cnt
       FROM rentals r JOIN members m ON r.member_id = m.id
       GROUP BY m.name
       HAVING COUNT(*) >= 3
       ORDER BY m.name""")

('Alice', 3)
('Bob', 3)
  → 2행


[('Alice', 3), ('Bob', 3)]

### JOIN 정리
- **핵심**: 표를 나눠 담아도(정규화) `JOIN`으로 언제든 다시 합쳐 봅니다. `INNER`=매칭만, `LEFT`=왼쪽 전부(+NULL).
- 세 표를 이어 회원명·도서명을 붙이고, `INNER JOIN`과 `LEFT JOIN`의 차이로 대여 0인 Emma 포함 여부를 확인합니다.
- **흔한 실수**: LEFT JOIN에서 `COUNT(*)`를 쓰면 Emma가 **0이 아니라 1**로 나옵니다(매칭 없는 행도 1행이므로). 반드시 **`COUNT(오른쪽표의 열)`**(`COUNT(r.id)`)로 세야 0이 됩니다.
- **범위**: RIGHT/FULL OUTER JOIN·서브쿼리는 여기서는 다루지 않고, INNER/LEFT까지 집중합니다.

## 인덱스 — 검색을 빠르게 하는 '찾아보기'

- **인덱스(index)** = 책 뒤의 **'찾아보기(색인)'**. 500쪽에서 특정 단어를 찾을 때, 처음부터 넘기기(전체 훑기=`SCAN`) vs 색인에서 페이지를 바로 찾기(`SEARCH`).
- 앞서 JOIN에서 계속 쓰던 `rentals` 표에 `CREATE INDEX` **한 줄**을 걸어, DB의 실행계획이 `SCAN`(전체 훑기) → `SEARCH ... USING INDEX`(색인 탐색)로 **바뀌는 것**을 눈으로 확인합니다.
- 📌 여기서는 **문법과 원리**에 집중합니다 — 표가 작아 실행 시간 차이는 눈에 안 띄지만, 표가 커질수록 이 '색인 탐색'이 검색 속도를 좌우합니다. (정성 결과 "색인이 있으면 훨씬 빠르다"만 확정)

In [91]:
# ✅ 설치 없이 실행(sqlite3 내장)
# ✅ 포인트: 인덱스가 없으면 DB는 표를 '처음부터 끝까지' 훑습니다(SCAN = 전체 훑기).
# 💡 EXPLAIN QUERY PLAN = 이 질의를 DB가 '어떻게' 처리하는지 미리 보기. (여기서는 SCAN/SEARCH 문구만 확인합니다.)
# 앞서 JOIN에서 계속 쓰던 member_id로 rentals를 찾는 질의를 예로 듭니다.
QUERY = "SELECT * FROM rentals WHERE member_id = 2"

plan_before = cur.execute("EXPLAIN QUERY PLAN " + QUERY).fetchall()
print("실행계획(인덱스 전):", plan_before[0][3])   # → 'SCAN rentals' = 전체 훑기

실행계획(인덱스 전): SEARCH rentals USING INDEX idx_member (member_id=?)


In [89]:
# ✅ 포인트: CREATE INDEX 한 줄이면 DB가 member_id '색인'을 만들어 바로 찾아갑니다(SEARCH USING INDEX).
# 💡 CREATE INDEX는 여러 RDBMS에서 공통적으로 쓰는 기본 인덱스 문법입니다.
cur.execute("CREATE INDEX idx_member ON rentals(member_id)")

plan_after = cur.execute("EXPLAIN QUERY PLAN " + QUERY).fetchall()
print("실행계획(인덱스 후):", plan_after[0][3])   # → 'SEARCH rentals USING INDEX idx_member (member_id=?)'
print("→ SCAN(전체 훑기)이 SEARCH(색인 탐색)로 바뀌었습니다. 표가 커질수록 이 차이가 검색 속도를 좌우합니다.")

실행계획(인덱스 후): SEARCH rentals USING INDEX idx_member (member_id=?)
→ SCAN(전체 훑기)이 SEARCH(색인 탐색)로 바뀌었습니다. 표가 커질수록 이 차이가 검색 속도를 좌우합니다.


In [ ]:
# 🧪 실습 — rentals의 book_id에 인덱스를 만들고 EXPLAIN QUERY PLAN 으로 확인하세요
# 💡 힌트: CREATE INDEX 이름 ON 표(열) — SCAN 이 SEARCH 로 바뀌면 성공
# 👇 아래 빈 셀에 직접 작성해 보세요

In [92]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# # ① 만들기 전
run("EXPLAIN QUERY PLAN SELECT * FROM rentals WHERE book_id = 1")
# # → SCAN rentals
#
# # ② 인덱스 만들기
cur.execute("DROP INDEX IF EXISTS idx_rentals_book")   # 재실행 안전
cur.execute("CREATE INDEX idx_rentals_book ON rentals(book_id)")
#
# # ③ 다시 확인
run("EXPLAIN QUERY PLAN SELECT * FROM rentals WHERE book_id = 1")
# # → SEARCH rentals USING INDEX idx_rentals_book (book_id=?)

(2, 0, 216, 'SCAN rentals')
  → 1행
(3, 0, 61, 'SEARCH rentals USING INDEX idx_rentals_book (book_id=?)')
  → 1행


[(3, 0, 61, 'SEARCH rentals USING INDEX idx_rentals_book (book_id=?)')]

In [ ]:
# 🧪 실습 — 종합 실습 — JOIN과 집계를 조합해 보세요
# 💡 힌트: Q1 Top3 / Q2 모든 회원(0건 포함) / Q3 세 표 + 미반납 조건
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ── 정답 (직접 해본 뒤 주석을 풀어 확인) ──
# Q1. 가장 많이 대여된 책 Top 3 (제목·횟수)
run("""SELECT b.title, COUNT(*) AS cnt FROM rentals r
        JOIN books b ON r.book_id = b.id
        GROUP BY b.id ORDER BY cnt DESC, b.title LIMIT 3""")

('The Catcher in the Rye', 3)
('1984', 2)
('To Kill a Mockingbird', 2)
  → 3행


[('The Catcher in the Rye', 3), ('1984', 2), ('To Kill a Mockingbird', 2)]

In [94]:
# # Q2. 모든 회원의 대여 건수 (0건 포함, 많은 순)
run("""SELECT m.name, COUNT(r.id) AS cnt FROM members m
        LEFT JOIN rentals r ON m.id = r.member_id
        GROUP BY m.id ORDER BY cnt DESC, m.name""")

('Alice', 3)
('Bob', 3)
('Charlie', 2)
('David', 2)
('Emma', 0)
  → 5행


[('Alice', 3), ('Bob', 3), ('Charlie', 2), ('David', 2), ('Emma', 0)]

In [95]:
# Q3. 아직 반납 안 한 책의 제목과 빌린 사람 이름
run("""SELECT b.title, m.name FROM rentals r
        JOIN books   b ON r.book_id   = b.id
        JOIN members m ON r.member_id = m.id
        WHERE r.return_date IS NULL
        ORDER BY b.title""")

('1984', 'Alice')
('Animal Farm', 'Bob')
('The Catcher in the Rye', 'Bob')
('To Kill a Mockingbird', 'Charlie')
  → 4행


[('1984', 'Alice'),
 ('Animal Farm', 'Bob'),
 ('The Catcher in the Rye', 'Bob'),
 ('To Kill a Mockingbird', 'Charlie')]

### 인덱스 정리 — 트레이드오프
- **핵심**: 인덱스 = 찾아보기. `EXPLAIN QUERY PLAN`이 `SCAN`(전체 훑기) → `SEARCH ... USING INDEX`(색인 탐색)로 바뀝니다. 표가 커질수록 이 '색인 탐색'이 대규모 검색을 훨씬 빠르게 해 줍니다.
- **트레이드오프(개념 한 줄)**: 인덱스는 **검색은 빠르게** 하지만, **넣기/고치기는 조금 느려지고 저장 공간**을 씁니다 — 그래서 모든 열에 무작정 걸지 않습니다.
- **범위**: 실행계획 자세히 읽기·B-tree 내부·복합 인덱스는 여기서는 다루지 않습니다.

In [ ]:
# 실습을 마치며 연결을 닫습니다.
conn.close()
print("연결을 닫았습니다.")

## 마무리

도서관 `library` DB로 **JOIN과 인덱스**의 기본기를 확인했습니다.
- **JOIN**: 나눠 담은 표를 다시 이어(INNER/LEFT) "누가 무슨 책을 빌렸나"를 한 줄로 조회합니다.
- **인덱스**: `CREATE INDEX` 한 줄로 실행계획이 전체 훑기(SCAN)에서 색인 탐색(SEARCH)으로 바뀝니다 — 대규모 검색을 빠르게 합니다. 단, 쓰기 성능과 저장 공간에는 트레이드오프가 있습니다.

RAG 챗봇이 답할 때 참고하는 **문서 원문·출처·권한·메타데이터**도 결국 표에 저장됩니다. 표를 만들고·꺼내고·잇고·빠르게 찾는 기본기가 RDBMS 활용의 출발점입니다.


---
# 7교시 · 종합 실습 — 쇼핑몰 DB를 처음부터

## 종합 연습 — 나만의 쇼핑몰 DB (shop.db)

마지막 도전이에요. 이번에는 **스키마(설계도)만** 드릴 테니, DB 생성부터 조회·조작·집계까지 오늘 배운 것만으로 직접 만들어 봅니다. (JOIN은 필요 없어요 — 모든 문제는 표 하나 안에서 풀립니다.)

**스키마 — 표 3개**

| 표 | 열 |
|---|---|
| `products` (상품) | `id` INTEGER **PK** · `name` TEXT · `category` TEXT · `price` INTEGER · `stock` INTEGER |
| `customers` (고객) | `id` INTEGER **PK** · `name` TEXT · `email` TEXT · `join_date` TEXT |
| `orders` (주문) | `id` INTEGER **PK** · `customer_id` INTEGER **FK**→`customers.id` · `product_id` INTEGER **FK**→`products.id` · `qty` INTEGER · `order_date` TEXT · `ship_date` TEXT (NULL이면 배송 전) |

![ERD](attachment:erd.png)

⚠️ **Q1부터 순서대로** 풀어 주세요 — 앞 문제의 변경(INSERT·UPDATE)이 뒤 문제 결과로 이어집니다. 정답은 맨 아래 **정답 모음**에 한꺼번에 있어요.

In [ ]:
# ✅ 제공 셀 — 그대로 실행: 쇼핑몰 DB(shop.db) 연결 + 출력 헬퍼 srun()
# 💡 library.db와 별개의 새 DB 파일입니다. (헷갈리지 않게 연결 변수도 sconn/scur로 따로 씁니다)
import sqlite3
sconn = sqlite3.connect("shop.db")
scur = sconn.cursor()

def srun(sql):
    rows = scur.execute(sql).fetchall()
    for r in rows:
        print(r)
    print(f"  → {len(rows)}행")
    return rows

print("shop.db 연결 완료")

In [ ]:
# 🧪 Q1 — 위 스키마(ERD)대로 표 3개를 만드세요 (products · customers · orders)
# 💡 힌트: scur.executescript("""...""") 로 여러 문장을 한 번에. 앞에 DROP TABLE IF EXISTS 를 두면 재실행 안전
# 💡 외래키는 FOREIGN KEY(customer_id) REFERENCES customers(id) 형태
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# ✅ 제공 셀 — Q1을 완료한 뒤 실행: 시드 데이터 적재 (표가 없으면 오류가 납니다 → Q1 먼저!)
# 💡 재실행해도 안전: 기존 행을 지우고 같은 시드를 다시 넣습니다
scur.execute("DELETE FROM orders")
scur.execute("DELETE FROM customers")
scur.execute("DELETE FROM products")
scur.executemany("INSERT INTO products VALUES (?,?,?,?,?)", [
    (1, "노트북",           "전자기기", 1500000,   5),
    (2, "무선 마우스",       "전자기기",   35000,  40),
    (3, "기계식 키보드",     "전자기기",  120000,  25),
    (4, "27인치 모니터",     "전자기기",  350000,  10),
    (5, "USB 허브",          "전자기기",   28000,  60),
    (6, "스테인리스 텀블러", "생활용품",   22000,  80),
    (7, "노트북 백팩",       "생활용품",   65000,  30),
    (8, "데스크 매트",       "문구",       15000, 100),
])
scur.executemany("INSERT INTO customers VALUES (?,?,?,?)", [
    (1, "김하늘", "haneul@example.com",  "20240105"),
    (2, "이준호", "junho@example.com",   "20240220"),
    (3, "박서연", "seoyeon@example.com", "20240314"),
    (4, "최민재", "minjae@example.com",  "20240502"),
    (5, "정유나", "yuna@example.com",    "20240711"),
])
scur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [
    (1,  1, 2, 1, "20250701", "20250702"),
    (2,  1, 3, 1, "20250703", "20250705"),
    (3,  2, 1, 1, "20250705", "20250708"),
    (4,  3, 6, 2, "20250706", "20250707"),
    (5,  2, 5, 3, "20250710", "20250711"),
    (6,  4, 8, 2, "20250712", None),
    (7,  1, 6, 1, "20250715", "20250716"),
    (8,  3, 4, 1, "20250718", None),
    (9,  3, 2, 2, "20250720", "20250721"),
    (10, 2, 7, 1, "20250722", None),
    (11, 4, 3, 1, "20250725", None),
    (12, 1, 5, 2, "20250728", "20250729"),
])
sconn.commit()
print("시드 적재 완료 → products 8 · customers 5 · orders 12")

In [ ]:
# 🧪 Q2 — 신상품 '웹캠'을 등록하세요 — (9, '웹캠', '전자기기', 45000, 20)
# 💡 힌트: INSERT INTO products VALUES (...) → sconn.commit() → srun 으로 SELECT 확인
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q3 — 가장 비싼 상품 Top 3의 이름과 가격을 조회하세요
# 💡 힌트: ORDER BY ... DESC 로 줄 세우고 LIMIT 으로 자릅니다
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q4 — 이름이 '노트북'으로 시작하는 상품의 이름·가격을 조회하세요
# 💡 힌트: LIKE 와 % (% = 아무 글자)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q5 — 가격이 2만 원 이상 5만 원 이하인 상품의 이름·가격을 조회하세요
# 💡 힌트: BETWEEN a AND b (양끝 포함) — Q2에서 등록한 웹캠도 나와야 해요
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q6 — 아직 배송 전인 주문의 id·product_id·order_date를 조회하세요
# 💡 힌트: '값이 없음'은 = NULL 이 아니라 IS NULL
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q7 — 주문한 적이 있는 고객 id를 중복 없이 조회하세요
# 💡 힌트: DISTINCT — 고객 5명 중 몇 명이 나올까요?
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q8 — 스테인리스 텀블러(id=6)의 가격을 25000원으로 바꾸세요 — 값은 placeholder ? 로 전달
# 💡 힌트: scur.execute("UPDATE ... SET price = ? WHERE id = ?", (값, id)) → commit → SELECT 확인
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q9 — 카테고리별 상품 수와 평균 가격을 조회하세요 (열 이름은 cnt·avg_price 로)
# 💡 힌트: GROUP BY 로 묶고 COUNT(*)·AVG(price) 로 요약, 별칭은 AS
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q10 — 주문을 2건 이상 한 고객 id와 건수를 많은 순으로 조회하세요
# 💡 힌트: GROUP BY → HAVING COUNT(*) >= 2 → ORDER BY cnt DESC (동점이면 customer_id 순)
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q11 — Q2에서 등록한 웹캠(id=9)을 삭제하고, 상품 수가 8로 돌아왔는지 확인하세요
# 💡 힌트: DELETE FROM ... WHERE → commit → srun("SELECT COUNT(*) FROM products")
# 👇 아래 빈 셀에 직접 작성해 보세요

### 정답 모음 — 직접 푼 뒤에 확인하세요

주석을 풀어 실행하면 각 셀의 `# →` 기대 결과와 같아야 합니다. (Q1부터 **순서대로** 풀었을 때 기준)

In [ ]:
# ── Q1 정답 ──
# # ⚠️ 실행하면 표가 새로 만들어져 비워집니다 — 실행 후 위 시드 셀을 다시 실행하세요
# scur.executescript("""
# DROP TABLE IF EXISTS orders;
# DROP TABLE IF EXISTS customers;
# DROP TABLE IF EXISTS products;
#
# CREATE TABLE products (
#     id INTEGER PRIMARY KEY, name TEXT, category TEXT, price INTEGER, stock INTEGER
# );
# CREATE TABLE customers (
#     id INTEGER PRIMARY KEY, name TEXT, email TEXT, join_date TEXT
# );
# CREATE TABLE orders (
#     id INTEGER PRIMARY KEY, customer_id INTEGER, product_id INTEGER,
#     qty INTEGER, order_date TEXT, ship_date TEXT,
#     FOREIGN KEY(customer_id) REFERENCES customers(id),
#     FOREIGN KEY(product_id) REFERENCES products(id)
# );
# """)
# print(scur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
# # → [('products',), ('customers',), ('orders',)]

In [ ]:
# ── Q2 정답 ──
# scur.execute("INSERT INTO products VALUES (9, '웹캠', '전자기기', 45000, 20)")
# sconn.commit()
# srun("SELECT * FROM products WHERE id = 9")
# # → (9, '웹캠', '전자기기', 45000, 20)  → 1행

In [ ]:
# ── Q3 정답 ──
# srun("SELECT name, price FROM products ORDER BY price DESC LIMIT 3")
# # → ('노트북', 1500000) ('27인치 모니터', 350000) ('기계식 키보드', 120000)  → 3행

In [ ]:
# ── Q4 정답 ──
# srun("SELECT name, price FROM products WHERE name LIKE '노트북%'")
# # → ('노트북', 1500000) ('노트북 백팩', 65000)  → 2행

In [ ]:
# ── Q5 정답 ──
# srun("SELECT name, price FROM products WHERE price BETWEEN 20000 AND 50000")
# # → ('무선 마우스', 35000) ('USB 허브', 28000) ('스테인리스 텀블러', 22000) ('웹캠', 45000)  → 4행
# # 💡 웹캠(45000)이 안 보이면 Q2를 건너뛴 거예요

In [ ]:
# ── Q6 정답 ──
# srun("SELECT id, product_id, order_date FROM orders WHERE ship_date IS NULL")
# # → (6, 8, '20250712') (8, 4, '20250718') (10, 7, '20250722') (11, 3, '20250725')  → 4행
# # ⚠️ = NULL 로 쓰면 오류 없이 0행이 나옵니다 — 꼭 IS NULL

In [ ]:
# ── Q7 정답 ──
# srun("SELECT DISTINCT customer_id FROM orders")
# # → (1,) (2,) (3,) (4,)  → 4행
# # 💡 5번 정유나 님은 주문이 없어 아예 안 나옵니다 → 내일 LEFT JOIN 에서 해결!

In [ ]:
# ── Q8 정답 ──
# scur.execute("UPDATE products SET price = ? WHERE id = ?", (25000, 6))
# sconn.commit()
# srun("SELECT name, price FROM products WHERE id = 6")
# # → ('스테인리스 텀블러', 25000)  → 1행

In [ ]:
# ── Q9 정답 ──
# srun("""SELECT category, COUNT(*) AS cnt, AVG(price) AS avg_price
#          FROM products GROUP BY category""")
# # → ('문구', 1, 15000.0) ('생활용품', 2, 45000.0) ('전자기기', 6, 346333.3333333333)  → 3행
# # 💡 생활용품 평균이 45000.0이 아니면 Q8(텀블러 25000원)을 건너뛴 거예요

In [ ]:
# ── Q10 정답 ──
# srun("""SELECT customer_id, COUNT(*) AS cnt FROM orders
#          GROUP BY customer_id
#          HAVING COUNT(*) >= 2
#          ORDER BY cnt DESC, customer_id""")
# # → (1, 4) (2, 3) (3, 3) (4, 2)  → 4행

In [ ]:
# ── Q11 정답 ──
# scur.execute("DELETE FROM products WHERE id = 9")
# sconn.commit()
# srun("SELECT COUNT(*) FROM products")
# # → (8,)  → 1행

## 종합 연습 — 쇼핑몰 DB를 JOIN으로 잇기 (shop.db, CP1~CP3에 이어서)

바로 위 CP1~CP3에서 쓰던 **`sconn`·`scur`·`srun`을 그대로 이어서** 씁니다 — **리셋하지 않습니다.**
그래서 텀블러(id=6) 가격은 CP3(Q8)에서 이미 25000원으로 바뀐 상태이고, 웹캠(id=9)은 CP3(Q11)에서
삭제되어 상품이 다시 8개인 상태 그대로 JOIN을 이어 붙입니다 — **CP4의 금액 합계가 CP3의 변경을
반영하는 것이 이 통합 노트북의 핵심 포인트**입니다 (교안 7교시 CP4 "흔한 실수" 참고).

> 📝 원래 두 노트북(`RDBMS_1`·`RDBMS_2`)은 각각 따로 열어도 실행되도록 이 지점에서 shop.db를
> 원본 상태로 다시 만드는 "제공 셀"이 있었습니다. 하나로 합친 이 노트북에서는 그 리셋 셀을
> 아래처럼 "이어받기 확인" 셀로 바꿨습니다 — CP1~CP3를 건너뛰고 여기부터 실행하면 표가 없어
> 오류가 나니, 반드시 위에서부터 순서대로 실행하세요.

⚠️ **문제는 순서대로** 풀어 주세요. 정답은 맨 아래 **정답 모음**에 한꺼번에 있어요.


In [ ]:
# ✅ 이어받기 확인 — CP1~CP3의 sconn·scur·srun을 그대로 씁니다 (리셋 아님)
srun("SELECT COUNT(*) FROM products")   # 8이어야 정상 (웹캠 삭제 반영)
srun("SELECT price FROM products WHERE id = 6")   # 25000이어야 정상 (CP3 가격 변경 반영)


In [ ]:
# 🧪 Q1 — 주문 목록에 상품 이름을 붙이세요 — 주문 id·상품명·수량, id 순 앞 5건
# 💡 힌트: orders o JOIN products p ON o.product_id = p.id → ORDER BY o.id LIMIT 5
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q2 — 세 표를 이어 '누가 무엇을 샀나'를 만드세요 — 주문 id·고객명·상품명·주문일, 주문일 순 앞 5건
# 💡 힌트: JOIN을 두 번 — customers c 와 products p 를 각각 잇습니다
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q3 — 아직 배송 전인 주문의 주문 id·고객명·상품명을 조회하세요
# 💡 힌트: 3표 JOIN + WHERE o.ship_date IS NULL
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q4 — 모든 고객의 주문 건수를 많은 순으로 조회하세요 — 주문이 없는 고객도 0으로!
# 💡 힌트: customers 기준 LEFT JOIN + COUNT(o.id) — COUNT(*)는 0을 1로 세는 함정!
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q5 — 고객별 총 주문 금액(가격 × 수량 합)을 많은 순으로 조회하세요
# 💡 힌트: 3표 JOIN + SUM(p.price * o.qty) AS total + GROUP BY
# 👇 아래 빈 셀에 직접 작성해 보세요

In [ ]:
# 🧪 Q6 — orders(product_id)에 인덱스를 만들고, 실행 계획이 SCAN → SEARCH로 바뀌는지 확인하세요
# 💡 힌트: EXPLAIN QUERY PLAN SELECT ... WHERE product_id = 2 → CREATE INDEX → 같은 EXPLAIN 다시
# 👇 아래 빈 셀에 직접 작성해 보세요

### 정답 모음 — 직접 푼 뒤에 확인하세요

주석을 풀어 실행하면 각 셀의 `# →` 기대 결과와 같아야 합니다. (제공 셀 실행 직후, Q1부터 순서대로 기준)

In [ ]:
# ── Q1 정답 ──
# srun("""SELECT o.id, p.name, o.qty FROM orders o
#         JOIN products p ON o.product_id = p.id
#         ORDER BY o.id LIMIT 5""")
# # → (1, '무선 마우스', 1) (2, '기계식 키보드', 1) (3, '노트북', 1)
# #   (4, '스테인리스 텀블러', 2) (5, 'USB 허브', 3)  → 5행

In [ ]:
# ── Q2 정답 ──
# srun("""SELECT o.id, c.name, p.name, o.order_date FROM orders o
#         JOIN customers c ON o.customer_id = c.id
#         JOIN products p ON o.product_id = p.id
#         ORDER BY o.order_date LIMIT 5""")
# # → (1, '김하늘', '무선 마우스', '20250701') (2, '김하늘', '기계식 키보드', '20250703')
# #   (3, '이준호', '노트북', '20250705') (4, '박서연', '스테인리스 텀블러', '20250706')
# #   (5, '이준호', 'USB 허브', '20250710')  → 5행

In [ ]:
# ── Q3 정답 ──
# srun("""SELECT o.id, c.name, p.name FROM orders o
#         JOIN customers c ON o.customer_id = c.id
#         JOIN products p ON o.product_id = p.id
#         WHERE o.ship_date IS NULL""")
# # → (6, '최민재', '데스크 매트') (8, '박서연', '27인치 모니터')
# #   (10, '이준호', '노트북 백팩') (11, '최민재', '기계식 키보드')  → 4행

In [ ]:
# ── Q4 정답 ──
# srun("""SELECT c.name, COUNT(o.id) AS cnt FROM customers c
#         LEFT JOIN orders o ON o.customer_id = c.id
#         GROUP BY c.id
#         ORDER BY cnt DESC, c.id""")
# # → ('김하늘', 4) ('이준호', 3) ('박서연', 3) ('최민재', 2) ('정유나', 0)  → 5행
# # 💡 1권 Q7(DISTINCT)에서 안 보이던 정유나 님이 드디어 0으로 나타납니다!

In [ ]:
# ── Q5 정답 ──
# srun("""SELECT c.name, SUM(p.price * o.qty) AS total FROM orders o
#         JOIN customers c ON o.customer_id = c.id
#         JOIN products p ON o.product_id = p.id
#         GROUP BY c.id
#         ORDER BY total DESC""")
# # → ('이준호', 1649000) ('박서연', 470000) ('김하늘', 236000) ('최민재', 150000)  → 4행
# #   (CP3에서 스테인리스 텀블러 가격을 22000 → 25000으로 바꾼 것이 반영된 값입니다)


In [ ]:
# ── Q6 정답 ──
# srun("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE product_id = 2")
# # → (..., 'SCAN orders')  ← 인덱스 전: 전부 훑기
# scur.execute("CREATE INDEX idx_orders_product ON orders(product_id)")
# srun("EXPLAIN QUERY PLAN SELECT * FROM orders WHERE product_id = 2")
# # → (..., 'SEARCH orders USING INDEX idx_orders_product (product_id=?)')  ← 점프!

---
# 8교시 · 개인 실습 — 한빛은행 + 정리

---
## ⚙️ 준비 — hanbit_bank 데이터 세팅 (실행만 하세요)

아래 셀을 실행하면 `hanbit_bank.db`가 만들어지고 3개 표에 데이터가 채워집니다.
- `customers`(고객 5명) · `accounts`(계좌 6개) · `transactions`(거래 8건)
- 표들은 **외래키로 연결**돼 있어요: `accounts.customer_id → customers.id`, `transactions.account_id → accounts.id`
- 여러 번 실행해도 안전합니다 (매번 지우고 새로 만듭니다).

In [ ]:
import sqlite3

# 결과(행)를 한 줄씩 보기 좋게 출력하는 도우미
def run(sql):
    for row in cur.execute(sql):
        print(row)

# hanbit_bank.db 생성 + 3개 표(고객·계좌·거래)에 데이터 적재
# (DROP 후 재생성 — 여러 번 실행해도 결과가 같습니다)
conn = sqlite3.connect("hanbit_bank.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS transactions;
DROP TABLE IF EXISTS accounts;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    id INTEGER PRIMARY KEY, name TEXT, grade TEXT, join_date TEXT   -- 고객: 이름·등급(일반/우수/VIP)·가입일
);
CREATE TABLE accounts (
    id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT, balance INTEGER,  -- 계좌: 소유고객·종류(입출금/정기예금/적금)·잔액(원)
    FOREIGN KEY(customer_id) REFERENCES customers(id)
);
CREATE TABLE transactions (
    id INTEGER PRIMARY KEY, account_id INTEGER, tx_type TEXT, amount INTEGER, tx_date TEXT,  -- 거래: 대상계좌·종류(입금/출금)·금액(원)·거래일
    FOREIGN KEY(account_id) REFERENCES accounts(id)
);
""")

cur.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", [
    (1, "김철수", "VIP",  "20230115"),
    (2, "이영희", "우수", "20230320"),
    (3, "박민수", "일반", "20240210"),
    (4, "최지은", "우수", "20240505"),
    (5, "정우성", "일반", "20250110"),
])
cur.executemany("INSERT INTO accounts VALUES (?, ?, ?, ?)", [
    (1, 1, "입출금",   3500000),
    (2, 1, "정기예금", 50000000),
    (3, 2, "입출금",   1200000),
    (4, 2, "적금",     8400000),
    (5, 3, "입출금",   450000),
    (6, 4, "적금",     3600000),
])
cur.executemany("INSERT INTO transactions VALUES (?, ?, ?, ?, ?)", [
    (1, 1, "입금", 2000000,  "20250601"),
    (2, 1, "출금", 500000,   "20250615"),
    (3, 2, "입금", 50000000, "20250110"),
    (4, 3, "입금", 1500000,  "20250620"),
    (5, 3, "출금", 300000,   "20250625"),
    (6, 4, "입금", 700000,   "20250701"),
    (7, 5, "출금", 150000,   "20250702"),
    (8, 6, "입금", 300000,   "20250703"),
])
conn.commit()
print("hanbit_bank.db 준비 완료 — 고객 5명 · 계좌 6개 · 거래 8건")

**🗺️ 한빛은행 DB의 ERD** — 고객 1명이 계좌 여러 개를, 계좌 1개가 거래 여러 건을 가집니다.

![ERD](attachment:erd.png)

---
## 🧪 실습 1. DB 연결 + 전체 조회 — "hanbit_bank에 접속하기"

**시나리오**: 데이터 분석의 첫걸음은 DB에 **연결**해 데이터가 잘 있는지 확인하는 것.
이번에는 `hanbit_bank.db`에 붙어 고객 목록을 꺼내 봅니다.

**요구사항**: `hanbit_bank.db`에 연결하고, `customers` 테이블 전체를 조회하는 코드를 작성하세요.

**기대 출력**: 고객 5명이 `(id, 이름, 등급, 가입일)` 형태로 출력 — 김철수(VIP)부터 정우성(일반)까지.

In [ ]:
# 🧪 hanbit_bank.db에 연결하고, customers 테이블 전체를 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 1)
- **연결**: library.db에 연결할 때 썼던 함수를 그대로 씁니다 — DB 파일 이름만 `"hanbit_bank.db"`로 바꾸면 됩니다.
- **커서·조회**: 연결(`conn`) 다음엔 `cur = conn.cursor()`로 커서를 만들고, `run("SELECT * FROM customers")`로 실행합니다.
- ⭐ `conn`은 DB로 가는 '전화선', `cur`(cursor)는 명령을 실어 나르는 '커서'라고 생각하면 쉬워요.

In [ ]:
# ── 실습 1 정답 ──
# run("SELECT * FROM customers")
# # → 고객 5명, (id, 이름, 등급, 가입일) — 김철수(VIP)부터 정우성(일반)까지


---
## 🧪 실습 2. WHERE — 우수 등급 고객 찾기

**시나리오**: 마케팅팀이 "**우수** 등급 고객 명단"을 요청했습니다. `customers`에서 등급이 '우수'인 고객만 골라 가입일 순으로 보여주세요.

**요구사항**: `customers`에서 등급(`grade`)이 '우수'인 고객만, 이름과 등급을 가입일(`join_date`) 순으로 조회하는 코드를 작성하세요.

**기대 출력**: `('이영희', '우수')`, `('최지은', '우수')` — 2명.

In [ ]:
# 🧪 등급이 '우수'인 고객만 이름·등급을 가입일 순으로 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 2)
- **패턴**: `SELECT 열목록 FROM customers WHERE 조건 ORDER BY 열` 형태입니다.
- **조건**: 등급 값은 세 가지(`일반`·`우수`·`VIP`) 중 하나를 고르면 됩니다.
- ⭐ SQL에서 **문자열 값은 반드시 작은따옴표**로 감쌉니다: `'우수'` (숫자는 따옴표 없이).

In [ ]:
# ── 실습 2 정답 ──
# run("SELECT name, grade FROM customers WHERE grade = '우수' ORDER BY join_date")
# # → ('이영희', '우수'), ('최지은', '우수')  → 2행


---
## 🧪 실습 3. ORDER BY + LIMIT — 잔액 상위 계좌

**시나리오**: "잔액이 가장 많은 계좌 TOP 3"를 뽑아 주세요. `accounts`를 잔액 **큰 순서**로 정렬한 뒤 위에서 3개만.

**요구사항**: `accounts`에서 잔액(`balance`)이 큰 순서로 상위 3개 계좌의 id·계좌종류·잔액을 조회하는 코드를 작성하세요.

**기대 출력**: `(2, '정기예금', 50000000)`, `(4, '적금', 8400000)`, `(6, '적금', 3600000)`.

In [ ]:
# 🧪 잔액이 큰 순서로 상위 3개 계좌(id·계좌종류·잔액)를 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 3)
- **패턴**: `ORDER BY 열 방향 LIMIT 개수`로 정렬한 뒤 일부만 봅니다.
- **방향**: 오름차순은 `ASC`, 내림차순은 `DESC`. 큰 잔액이 위로 오려면 어느 쪽일까요?
- ⭐ `LIMIT 3`은 "정렬한 결과의 위에서 3개만"이라는 뜻입니다.

In [ ]:
# ── 실습 3 정답 ──
# run("SELECT id, account_type, balance FROM accounts ORDER BY balance DESC LIMIT 3")
# # → (2,'정기예금',50000000), (4,'적금',8400000), (6,'적금',3600000)


---
## 🧪 실습 4. GROUP BY + SUM — 계좌 종류별 잔액 합계

**시나리오**: "입출금·정기예금·적금, **종류별로 잔액을 모두 더하면** 얼마씩일까?" `accounts`를 계좌 종류로 묶어 잔액 합계를 구합니다.

**요구사항**: `accounts`를 계좌종류(`account_type`)별로 묶어 잔액 합계(`total`)를 큰 순서로 조회하는 코드를 작성하세요.

**기대 출력**: `('정기예금', 50000000)`, `('적금', 12000000)`, `('입출금', 5150000)`.

In [ ]:
# 🧪 계좌종류별 잔액 합계(total)를 큰 순서로 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 4)
- **패턴**: `GROUP BY`로 같은 종류끼리 묶고, 집계 함수로 각 그룹의 값을 계산합니다.
- **집계함수**: 여러 값을 더하는 함수가 필요합니다. `AS total`로 이름 붙이는 것도 잊지 마세요.
- ⭐ 규칙: `GROUP BY`로 묶을 땐 SELECT의 **비집계 열**과 **GROUP BY 열이 일치**해야 합니다.

In [ ]:
# ── 실습 4 정답 ──
# run("SELECT account_type, SUM(balance) AS total FROM accounts "
#     "GROUP BY account_type ORDER BY total DESC")
# # → ('정기예금',50000000), ('적금',12000000), ('입출금',5150000)


---
## 🧪 실습 5. JOIN — VIP 고객의 총 잔액 ⭐

**시나리오**: "VIP 고객이 **가진 돈이 다 합쳐 얼마**인지" 알고 싶습니다. 그런데 이름은 `customers`에, 잔액은 `accounts`에 흩어져 있어요. 두 표를 **외래키로 이어(JOIN)** 합쳐야 답할 수 있습니다.

**요구사항**: `customers`와 `accounts`를 이어서, VIP 등급 고객의 이름별 잔액 합계(`total`)를 조회하는 코드를 작성하세요.

**기대 출력**: `('김철수', 53500000)` — 김철수의 입출금 350만 + 정기예금 5,000만 = **5,350만원**.

In [ ]:
# 🧪 VIP 등급 고객의 이름별 잔액 합계(total)를 조회하세요 (customers·accounts JOIN)
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 5)
- **패턴**: `FROM customers c JOIN accounts a ON c.id = a.___`로 두 표를 잇고, `WHERE`로 등급을 거른 뒤 `GROUP BY`로 묶습니다.
- **외래키**: `accounts`가 어떤 고객 것인지 가리키는 열이 `ON` 뒤에 옵니다.
- ⭐ **왜 JOIN?** 정규화로 이름(customers)과 잔액(accounts)을 나눠 담았기 때문에, 함께 보려면 다시 이어야 합니다.

In [ ]:
# ── 실습 5 정답 ──
# run("SELECT c.name, SUM(a.balance) AS total FROM customers c "
#     "JOIN accounts a ON c.id = a.customer_id "
#     "WHERE c.grade = 'VIP' GROUP BY c.name")
# # → ('김철수', 53500000) — 입출금 350만 + 정기예금 5,000만


---
## 🧪 실습 6. JOIN 한 번 더 — 고객별 거래 건수

**시나리오**: "고객마다 **거래를 몇 번** 했나?" 고객→계좌→거래가 사슬처럼 연결돼 있으니 **세 표를 모두 이어야** 셀 수 있습니다.

**요구사항**: `customers`·`accounts`·`transactions` 세 표를 이어서, 고객별 거래 건수(`tx_cnt`)를 많은 순서로 조회하는 코드를 작성하세요.

**기대 출력**: `('김철수', 3)`, `('이영희', 3)`, `('박민수', 1)`, `('최지은', 1)`. (계좌가 없는 정우성은 거래도 없어 결과에서 빠집니다.)

In [ ]:
# 🧪 세 표(customers·accounts·transactions)를 이어 고객별 거래 건수(tx_cnt)를 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 6)
- **패턴**: 표 3개를 `JOIN ... ON ...`으로 사슬처럼 잇습니다. 고객 → 계좌 → 거래 순서입니다.
- **외래키**: `transactions`에서 "이 거래가 어느 계좌 것인가"를 가리키는 열이 마지막 `ON`에 옵니다.
- ⭐ `COUNT(t.id)`로 거래 행의 개수를 셉니다. 3개 표를 이어야 '고객 이름'과 '거래 건수'를 한 줄에 놓을 수 있어요.

In [ ]:
# ── 실습 6 정답 ──
# run("SELECT c.name, COUNT(t.id) AS tx_cnt FROM customers c "
#     "JOIN accounts a ON c.id = a.customer_id "
#     "JOIN transactions t ON a.id = t.account_id "
#     "GROUP BY c.id ORDER BY tx_cnt DESC, c.id")
# # → ('김철수',3), ('이영희',3), ('박민수',1), ('최지은',1)
# #   (계좌가 없는 정우성은 거래도 없어 결과에서 빠집니다)


---
## 🧪 실습 7. HAVING — 총잔액 2천만원 이상 고객

**시나리오**: "총 잔액이 **2천만원 이상**인 큰손 고객만" 뽑고 싶습니다. 그런데 조건이 걸리는 대상은 개별 행이 아니라 **합계(SUM)** — 이럴 땐 `WHERE`가 아니라 다른 키워드를 씁니다.

**요구사항**: `customers`와 `accounts`를 이어 고객별 잔액 합계를 구하고, 합계가 2천만원(20000000) 이상인 고객만 남기는 코드를 작성하세요.

**기대 출력**: `('김철수', 53500000)` — 2천만 이상은 김철수뿐.

In [ ]:
# 🧪 customers·accounts를 이어 고객별 잔액 합계가 2천만원 이상인 고객만 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (실습 7)
- **패턴**: 개별 행 조건은 `WHERE`, 그룹으로 묶은 뒤의 집계 결과 조건은 다른 키워드를 씁니다.
- **위치**: `GROUP BY` **뒤**, `ORDER BY` **앞**에 옵니다 — `SUM(a.balance) >= 20000000`처럼요.
- ⭐ 순서 기억법: `WHERE`(고르기) → `GROUP BY`(묶기) → 집계 → **?**(묶은 결과 고르기) → `ORDER BY`(줄세우기).

In [ ]:
# ── 실습 7 정답 ──
# run("SELECT c.name, SUM(a.balance) AS total FROM customers c "
#     "JOIN accounts a ON c.id = a.customer_id "
#     "GROUP BY c.name HAVING SUM(a.balance) >= 20000000")
# # → ('김철수', 53500000) — 2천만 이상은 김철수뿐


---
## 🚀 추가 도전. LEFT JOIN — 계좌 0인 고객까지 포함

**시나리오**: "고객별 **계좌 개수**"를 세는데, **계좌가 하나도 없는 고객(정우성)도 0으로 표시**하고 싶습니다. 그냥 `JOIN`이면 정우성이 사라져요. 기준이 되는 고객 표를 **전부** 남기려면 어떻게 해야 할까요?

**요구사항**: 계좌가 없는 고객(정우성)까지 포함해서, 고객별 계좌 개수(`acc_cnt`)를 조회하는 코드를 작성하세요.

**기대 출력**: `('김철수', 2)`, `('이영희', 2)`, `('박민수', 1)`, `('최지은', 1)`, `('정우성', 0)` — 정우성이 **0**으로 나오면 성공!

In [ ]:
# 🧪 계좌 없는 고객(정우성)까지 포함해 고객별 계좌 개수(acc_cnt)를 조회하세요
# 👇 아래에 직접 작성해 보세요


### 💡 힌트 (추가 도전)
- **방향**: 기준이 되는 왼쪽 표(`customers`)를 전부 남기고, 오른쪽 표(`accounts`)는 있으면 붙입니다.
- **JOIN 종류**: 왼쪽 표를 전부 남기고 오른쪽은 있으면 붙이는 JOIN이 무엇이었죠? (일반 JOIN 앞에 방향을 붙입니다)
- ⭐ 필수 실습은 아니지만, "없는 데이터도 0으로 남기는 조회"가 필요할 때 자주 쓰는 패턴입니다.

In [ ]:
# ── 추가 도전 정답 ──
# run("SELECT c.name, COUNT(a.id) AS acc_cnt FROM customers c "
#     "LEFT JOIN accounts a ON c.id = a.customer_id "
#     "GROUP BY c.name ORDER BY acc_cnt DESC, c.name")
# # → ('김철수',2), ('이영희',2), ('박민수',1), ('최지은',1), ('정우성',0)
# #   정우성이 0으로 나오면 성공!


---
## 🎉 마무리

여러분이 한빛은행 데이터로 직접 완성한 것:
- **조회**: `SELECT` + `WHERE`로 우수 등급 고객, `ORDER BY`+`LIMIT`으로 잔액 TOP 계좌
- **집계**: `GROUP BY` + `SUM`으로 종류별 잔액 합계
- **연결**: `JOIN`으로 VIP 총잔액·고객별 거래 건수 확인
- **조건**: `HAVING`으로 집계 결과에 조건 걸기
- **추가 도전**: `LEFT JOIN`으로 계좌 0인 정우성까지 포함

### 🤔 돌아보기 질문
1. 엑셀 대신 DB를 쓰는 이유는? (중복·정합성·동시 사용·검색 속도)
2. 기본키와 외래키의 차이는? (`customers.id` ↔ `accounts.customer_id`)
3. `WHERE`와 `HAVING`은 각각 언제 쓰나요? (집계 전 / 집계 후)
4. 여러 표에 나뉜 데이터를 함께 보려면 어떤 기준 열을 확인해야 할까요?
5. 한빛은행 데이터로 작성한 SQL을 다른 업무 데이터에 적용하려면 무엇을 먼저 확인해야 할까요?

> 📌 핵심: 표 구조와 컬럼 이름은 바뀌어도, **조회 → 조건 → 정렬 → 집계 → JOIN** 흐름은 그대로 활용할 수 있습니다.


---
## 오늘 배운 것 — 전체 정리

| 개념 | 한 줄 요약 |
|---|---|
| DB API | `connect → cursor → execute → fetch`가 sqlite3의 기본 흐름, placeholder(`?`)로 값을 안전하게 전달 |
| 조회 | `SELECT` / `WHERE`(고르기, `IS NULL` 주의) / `ORDER BY`(정렬) / `LIMIT`(N개) |
| 조작 | `INSERT`·`UPDATE`·`DELETE` + `commit` — 항상 `WHERE`로 대상 지정, 조작은 이후 모든 조회에 영향을 남긴다 |
| 집계 | `COUNT·SUM·AVG·MIN·MAX` / `GROUP BY`(묶기) / `HAVING`(묶은 뒤 조건, WHERE와 구분) |
| JOIN | `INNER`=매칭만, `LEFT`=왼쪽 전부(+ 없으면 NULL) — LEFT 집계는 `COUNT(오른쪽표.열)` |
| 인덱스 | `CREATE INDEX` 한 줄로 실행계획이 `SCAN`(전체 훑기) → `SEARCH`(색인 탐색)로 바뀐다 |

자세한 서술형 설명·흔한 실수·연결 고리는 `SQL_기초.md` 교안을 함께 보세요.
